# WTI Oil Price Forecasting — Prophet vs. Analyst Agent

> **Part 2 of 2.** This notebook builds on the baseline established in [Notebook 1 — Oil Prices in 2026](01_energy_oil_case_study.ipynb), where we showed that Prophet — a strong statistical model — was caught completely off-guard by the geopolitical shock of early 2026.  Here, we ask whether an agent that can *read the news* can do better.

This notebook asks a single question in two ways:

1. **Trajectory** — where will WTI crude be in 5, 10, and 21 business days?
2. **Binary** — will WTI close more than \$5/bbl *higher* than today within 5 trading days?

We compare two approaches across both tasks:

| Method | What it knows | How it forecasts |
|---|---|---|
| **Prophet** | Historical prices only | Statistical trend + seasonality |
| **Analyst Agent** | Prices + live news context | Context Agent → Analyst Agent (LLM) |

The period studied (Jan–Mar 2026) includes the geopolitical shock that pushed WTI from \$62 to \$95+.
Prophet cannot see it coming. An agent that reads the news can.


In [1]:
from __future__ import annotations

import json
import logging
import os
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as psp
import scipy.stats
from dotenv import load_dotenv
from IPython.display import Markdown as MD  # noqa: N814
from IPython.display import display  # noqa: A004


warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)

# ── Repo root ─────────────────────────────────────────────────────────────
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = _cwd
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"

for p in [str(REPO_ROOT / "implementations"), str(REPO_ROOT / "aieng-forecasting")]:
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(REPO_ROOT / ".env", override=False)

# ── Colour palette ────────────────────────────────────────────────────────
CLR_ACTUAL = "#2171b5"  # blue  — truth
CLR_HISTORY = "#bdd7e7"  # light blue — history shading
CLR_PROPHET = "#636363"  # grey  — Prophet
CLR_AGENT = "#2ca02c"  # green — Analyst Agent
CLR_CONFLICT = "#d62728"  # red   — shock / conflict annotation

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print("Setup complete.")

Repo root : /Users/ethanjackson/agentic-forecasting
Data dir  : /Users/ethanjackson/agentic-forecasting/data
Setup complete.


In [2]:
# ── WTI price history ────────────────────────────────────────────────────
PRICE_CACHE = DATA_DIR / "wti_price_history.parquet"

price_df = pd.read_parquet(PRICE_CACHE)
price_df.index = pd.DatetimeIndex([pd.Timestamp(str(d)[:10]) for d in price_df.index])
price_df.index.name = "date"
price_df = price_df.sort_index()

# ── Origins ───────────────────────────────────────────────────────────────
# Three dates that tell a story: calm → rising tension → active conflict.
TRAJECTORY_ORIGINS = [
    pd.Timestamp("2026-01-05"),  # Jan: calm, prices range-bound ~$60
    pd.Timestamp("2026-02-02"),  # Feb: early tension signals, $62
    pd.Timestamp("2026-03-02"),  # Mar: conflict just erupted, $71
]

# 8 weekly origins for the binary experiment.
SHOCK_ORIGINS = [
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-02-09"),
    pd.Timestamp("2026-02-17"),
    pd.Timestamp("2026-02-23"),
    pd.Timestamp("2026-03-02"),
    pd.Timestamp("2026-03-09"),
    pd.Timestamp("2026-03-16"),
    pd.Timestamp("2026-03-23"),
]

print(f"WTI price history : {price_df.index[0].date()} → {price_df.index[-1].date()} ({len(price_df):,} days)")
print("Trajectory origins:", [o.strftime("%Y-%m-%d") for o in TRAJECTORY_ORIGINS])
print("Shock origins     :", [o.strftime("%Y-%m-%d") for o in SHOCK_ORIGINS])

WTI price history : 2021-01-04 → 2026-05-01 (1,340 days)
Trajectory origins: ['2026-01-05', '2026-02-02', '2026-03-02']
Shock origins     : ['2026-02-02', '2026-02-09', '2026-02-17', '2026-02-23', '2026-03-02', '2026-03-09', '2026-03-16', '2026-03-23']


---
## Step 1 — Prophet Baseline

We start with Prophet, Meta's open-source time series model. It fits a trend + seasonality decomposition and extrapolates forward.  It has no access to news or events — every forecast is derived entirely from past prices.

We run it at each of our origins and store the 21-business-day trajectory (fan chart) so we can compare it directly to the agent later.

In [3]:
from prophet import Prophet  # type: ignore[import-untyped]  # noqa: E402


PROPHET_TRAJ_CACHE = DATA_DIR / "energy_prophet_trajectories.parquet"
PROPHET_SHOCK_TRAJ_CACHE = DATA_DIR / "energy_shock_prophet_trajectories.parquet"


def _fit_prophet_at_origin(price_df: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    """Fit Prophet on history up to origin; return 21-business-day trajectory."""
    train_df = price_df.loc[:origin][["price"]].reset_index()
    train_df.columns = pd.Index(["ds", "y"])

    model = Prophet(
        interval_width=0.95,
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=True,
        seasonality_mode="multiplicative",
    )
    model.fit(train_df)

    future = model.make_future_dataframe(periods=35, freq="D")
    pred = model.predict(future).set_index("ds")

    bday_dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=21)
    rows = []
    for h, date in enumerate(bday_dates, start=1):
        cal_date = date.normalize()
        if cal_date in pred.index:
            row = pred.loc[cal_date]
        else:
            nearest_idx = int((pred.index - cal_date).abs().argmin())
            row = pred.iloc[nearest_idx]
        rows.append(
            {
                "origin": origin,
                "forecast_date": date,
                "horizon": h,
                "yhat": float(row["yhat"]),
                "yhat_lower": float(row["yhat_lower"]),
                "yhat_upper": float(row["yhat_upper"]),
            }
        )
    return pd.DataFrame(rows)


def load_prophet_trajectories(
    price_df: pd.DataFrame,
    origins: list[pd.Timestamp],
    cache_path: Path,
) -> pd.DataFrame:
    """Load from cache or fit Prophet at each origin."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"] = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} Prophet trajectory rows from {cache_path.name}")
        return df
    print(f"Fitting Prophet at {len(origins)} origins ...")
    frames = []
    for origin in origins:
        print(f"  {origin.date()} ...", end=" ", flush=True)
        frames.append(_fit_prophet_at_origin(price_df, origin))
        print("done")
    df = pd.concat(frames, ignore_index=True)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path.name}")
    return df


prophet_traj_df = load_prophet_trajectories(price_df, TRAJECTORY_ORIGINS, PROPHET_TRAJ_CACHE)
prophet_shock_traj_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(f"Trajectory prophet: {len(prophet_traj_df)} rows")
print(f"Shock prophet     : {len(prophet_shock_traj_df)} rows")

Loaded 63 Prophet trajectory rows from energy_prophet_trajectories.parquet
Loaded 168 Prophet trajectory rows from energy_shock_prophet_trajectories.parquet
Trajectory prophet: 63 rows
Shock prophet     : 168 rows


---
## Step 2 — Introducing the Analyst Agent

Instead of another statistical model, we build a **two-stage agent pipeline** where each stage has a distinct, separable role:

---

### Context Agent
*Google ADK · gemini-3-flash-preview · google\_search*

Searches the web for oil market intelligence as of the prediction date. A **strict temporal cutoff is enforced via system prompt** — only information available *before* the origin date is included; post-cutoff results are discarded. Returns a concise structured markdown briefing covering supply risks, OPEC+ policy, Persian Gulf geopolitics, and analyst price targets.

⬇️ *oil market briefing (markdown)*

---

### Analyst Agent
*litellm · gemini/gemini-3-flash-preview*

Receives the price history and market briefing. The **system prompt never changes** — it simply instructs the agent to read the data and execute whatever task is specified in the user message. The task specification defines the question *and* the required JSON output schema, making the agent trivially extensible to any new question type without touching the agent configuration.

| Task | Question | Output |
|---|---|---|
| **A — Trajectory** | Where will WTI be in 5 / 10 / 21 business days? | Point estimates + 80% confidence intervals |
| **B — Binary** | Will WTI close > today + \$5 in 5 trading days? | Probability, direction bias, key signals |
| **C — Scenarios** | What are the top 3 analyst scenarios for WTI over the next 60 days? | Scenario cards with probabilities + conditional price estimates |

---

This separation is the core design principle: **context retrieval and analytical reasoning are independently improvable**. And because the task is just text in a user message, the same agent configuration handles prediction tasks, probability estimation, and open-ended qualitative analysis.

In [4]:
import litellm  # noqa: E402
from google.adk.agents import LlmAgent  # noqa: E402
from google.adk.runners import Runner  # noqa: E402
from google.adk.sessions import InMemorySessionService  # noqa: E402
from google.adk.tools import google_search  # noqa: E402
from google.genai import types as genai_types  # noqa: E402
from google.genai.types import GenerateContentConfig  # noqa: E402


# ── Cache paths ───────────────────────────────────────────────────────────────
TRAJ_AGENT_CACHE = DATA_DIR / "energy_agent_trajectory_forecasts.json"
TRAJ_CONTEXT_CACHE = DATA_DIR / "energy_agent_trajectory_context.json"
SHOCK_ANALYST_CACHE = DATA_DIR / "energy_upshock_analyst_forecasts.json"
SHOCK_CONTEXT_CACHE = DATA_DIR / "energy_upshock_news_context.json"
SCENARIO_CACHE = DATA_DIR / "energy_agent_scenario_forecasts.json"

# ── Task parameters ───────────────────────────────────────────────────────────
SHOCK_THRESHOLD = 5.0  # $5/bbl upward move
SHOCK_HORIZON = 5  # business days


# ── History helpers ───────────────────────────────────────────────────────────
def compress_history(price_df: pd.DataFrame, as_of: pd.Timestamp, recent_window_months: int = 6) -> pd.DataFrame:
    """Weekly averages for older data, daily for recent 6 months."""
    hist = price_df[price_df.index <= as_of].copy()
    cutoff = as_of - pd.DateOffset(months=recent_window_months)
    older = (
        hist[hist.index < cutoff]
        .resample("W")
        .mean()
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )
    recent = hist[hist.index >= cutoff].reset_index().rename(columns={"date": "timestamp", "price": "value"})
    result = pd.concat([older, recent], ignore_index=True)
    result["timestamp"] = pd.to_datetime(result["timestamp"])
    return result[["timestamp", "value"]].sort_values("timestamp").reset_index(drop=True)


def serialize_history(df: pd.DataFrame, precision: int = 2) -> str:
    """Serialise compressed history to a compact dated string."""
    return "\n".join(
        f"{row['timestamp'].strftime('%Y-%m-%d')} ${row['value']:.{precision}f}" for _, row in df.iterrows()
    )


# ─────────────────────────────────────────────────────────────────────────────
# Context Agent (Google ADK + google_search)
# ─────────────────────────────────────────────────────────────────────────────
_CONTEXT_SYSTEM = (
    "You are an oil market intelligence specialist with access to web search.\n\n"
    "CRITICAL TEMPORAL CONSTRAINT — you are simulating the perspective of an analyst\n"
    "as of {cutoff_date}.\n"
    "- Include ONLY information publicly available BEFORE {cutoff_date}.\n"
    "- EXCLUDE any events, market moves, or data from {cutoff_date} or later.\n"
    "- If a search result post-dates the cutoff, skip it entirely.\n\n"
    "Search for and summarise:\n"
    "- WTI/Brent crude price level and recent trend\n"
    "- OPEC+ production decisions and supply outlook\n"
    "- Geopolitical risks in the Persian Gulf, Middle East, key shipping lanes\n"
    "- US Strategic Petroleum Reserve and energy policy signals\n"
    "- Notable tanker/shipping incidents or supply chain disruption signals\n"
    "- Published analyst forecasts or unusual price-target revisions\n\n"
    "Return a concise structured markdown summary (3-5 paragraphs)."
)


async def retrieve_context(cutoff_date: str) -> str:
    """Run the Context Agent; return oil-market briefing as of cutoff_date."""
    _app = "oil-context-agent"
    session_svc = InMemorySessionService()
    agent = LlmAgent(
        name="oil_context_agent",
        instruction=_CONTEXT_SYSTEM.format(cutoff_date=cutoff_date),
        tools=[google_search],
        model="gemini-3-flash-preview",
        generate_content_config=GenerateContentConfig(temperature=0.1, max_output_tokens=4096),
    )
    runner = Runner(agent=agent, app_name=_app, session_service=session_svc)
    session = await session_svc.create_session(app_name=_app, user_id="nb")
    prompt = (
        f"Oil market intelligence briefing as of {cutoff_date}. "
        f"Focus on supply risks, OPEC+ policy, Persian Gulf geopolitics, and factors "
        f"that could cause a sudden large price move in WTI. "
        f"IMPORTANT: only use information available before {cutoff_date}."
    )
    content = genai_types.Content(role="user", parts=[genai_types.Part(text=prompt)])
    async for event in runner.run_async(user_id="nb", session_id=session.id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text or ""
    return ""


# ─────────────────────────────────────────────────────────────────────────────
# Analyst Agent — single configuration, task specified in the user message
# ─────────────────────────────────────────────────────────────────────────────

_ANALYST_SYSTEM = (
    "You are an expert oil market analyst.\n\n"
    "You will receive:\n"
    "  1. WTI crude oil price history (daily, most recent last)\n"
    "  2. An oil market intelligence briefing with a strict temporal cutoff\n"
    "  3. A TASK SPECIFICATION that defines the exact question and the required\n"
    "     JSON output schema\n\n"
    "Read the data and briefing carefully, then execute the task precisely.\n"
    "Return ONLY valid JSON that matches the schema described in the task specification.\n"
    "Output ONLY the JSON object — no preamble, no explanation outside the JSON."
)

# ── Task A: Trajectory forecast ───────────────────────────────────────────────
TASK_TRAJECTORY = (
    "Forecast the WTI crude oil price at three forward horizons from today:\n"
    "  - 5  business days (~1 trading week)\n"
    "  - 10 business days (~2 trading weeks)\n"
    "  - 21 business days (~1 calendar month)\n\n"
    "For each horizon provide a point estimate and an 80% confidence interval.\n"
    "Be calibrated: historical weekly vol is roughly $2-5/bbl; widen intervals\n"
    "when the market is unusually uncertain.\n\n"
    "Return JSON with exactly these fields:\n"
    "{\n"
    '  "day_5":           <float>,\n'
    '  "lower_80_day_5":  <float>,\n'
    '  "upper_80_day_5":  <float>,\n'
    '  "day_10":          <float>,\n'
    '  "lower_80_day_10": <float>,\n'
    '  "upper_80_day_10": <float>,\n'
    '  "day_21":          <float>,\n'
    '  "lower_80_day_21": <float>,\n'
    '  "upper_80_day_21": <float>,\n'
    '  "reasoning":       "<2-4 sentences>",\n'
    '  "confidence":      "<high|medium|low>"\n'
    "}"
)

# ── Task B: Binary shock probability ─────────────────────────────────────────
TASK_SHOCK = (
    f"Estimate P(up) — the probability that WTI will close MORE THAN\n"
    f"${int(SHOCK_THRESHOLD)}/bbl HIGHER than today's price at the end of\n"
    f"{SHOCK_HORIZON} trading days.\n\n"
    "This is a directional upside question only.\n\n"
    "Calibration guidance:\n"
    "  - No unusual upside catalyst       -> base rate ~10-15%\n"
    "  - Escalating unconfirmed risk      -> 20-40%\n"
    "  - Confirmed supply disruption      -> 60-85%\n\n"
    "Return JSON with exactly these fields:\n"
    "{\n"
    '  "probability_up":  <float 0-1>,\n'
    '  "direction_bias":  "<up|down|neutral>",\n'
    '  "reasoning":       "<2-4 sentences>",\n'
    '  "key_signals":     ["<signal 1>", "<signal 2>", "<signal 3>"],\n'
    '  "confidence":      "<high|medium|low>"\n'
    "}"
)

# ── Task C: Scenario analysis ─────────────────────────────────────────────────
TASK_SCENARIOS = (
    "Identify the three scenarios that oil market analysts and experts are most\n"
    "actively debating for WTI crude over the next 60 days, given the current\n"
    "market context and price history.\n\n"
    "For each scenario:\n"
    "  - Give it a concise name (3-6 words)\n"
    "  - Describe it in 1-2 sentences\n"
    "  - Assign a probability (all three must sum to <= 1.0)\n"
    "  - Provide an expected WTI price range at the 60-day horizon as [low, high]\n"
    "  - Give your point estimate for WTI at 60 days under this scenario\n"
    "  - List 1-2 key drivers that would cause this scenario to materialise\n\n"
    "Also identify which scenario is the base case and provide an overall\n"
    "one-paragraph reasoning summary.\n\n"
    "Return JSON with exactly these fields:\n"
    "{\n"
    '  "scenarios": [\n'
    "    {\n"
    '      "name":               "<string>",\n'
    '      "description":        "<string>",\n'
    '      "probability":        <float>,\n'
    '      "wti_range_60d":      [<float_low>, <float_high>],\n'
    '      "point_estimate_60d": <float>,\n'
    '      "key_drivers":        ["<driver 1>", "<driver 2>"]\n'
    "    }\n"
    "  ],\n"
    '  "base_case":  "<scenario name>",\n'
    '  "reasoning":  "<paragraph>"\n'
    "}"
)


# ── Single run_analyst() ──────────────────────────────────────────────────────
def run_analyst(
    task_spec: str,
    history_str: str,
    context: str,
    origin_date: str,
    origin_price: float,
) -> dict:
    """Run the Analyst Agent for any task; task_spec defines the question + schema."""
    user_prompt = (
        f"### WTI Price History (ending {origin_date})\n\n{history_str}\n\n"
        f"---\n### Oil Market Briefing (as of {origin_date})\n\n{context}\n\n"
        f"---\n### TASK SPECIFICATION\n\n{task_spec}\n\n"
        f"Current WTI price: ${origin_price:.2f}/bbl on {origin_date}."
    )
    run_analyst._last_system = _ANALYST_SYSTEM
    run_analyst._last_user_prompt = user_prompt
    response = litellm.completion(
        model="gemini/gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": _ANALYST_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content or "{}"
    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip())
    raw = re.sub(r"\s*```$", "", raw)
    run_analyst._last_raw = raw
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"_parse_error": raw[:500]}


# ── Outcome helpers ───────────────────────────────────────────────────────────
def check_shock_outcome(
    price_df: pd.DataFrame, origin: pd.Timestamp, threshold: float, horizon_bdays: int
) -> tuple[int, float]:
    """(outcome, delta): outcome=1 if day-H close > origin_price + threshold."""
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    future = price_df[price_df.index > origin].iloc[:horizon_bdays]
    delta = float(future.iloc[-1]["price"]) - origin_price
    return (1 if delta > threshold else 0), delta


def prophet_prob_shock(
    prophet_traj_sub: pd.DataFrame, origin_price: float, threshold: float, horizon: int = 5
) -> float:
    """P(price_h > origin + threshold) from Prophet 95% CI (Gaussian approx)."""
    row = prophet_traj_sub[prophet_traj_sub["horizon"] == horizon]
    if row.empty:
        return float("nan")
    row = row.iloc[0]
    sigma = (float(row["yhat_upper"]) - float(row["yhat_lower"])) / (2 * 1.96)
    if sigma <= 0:
        return 1.0 if float(row["yhat"]) > origin_price + threshold else 0.0
    return float(
        np.clip(
            1.0 - scipy.stats.norm.cdf(origin_price + threshold, loc=float(row["yhat"]), scale=sigma),
            0.0,
            1.0,
        )
    )


print("Unified Analyst Agent loaded.")
print("  System prompt  : _ANALYST_SYSTEM (shared across all tasks)")
print("  Task A         : TASK_TRAJECTORY — point estimates at 5/10/21 bdays")
print(f"  Task B         : TASK_SHOCK      — P(up > +${SHOCK_THRESHOLD:.0f}) in {SHOCK_HORIZON} bdays")
print("  Task C         : TASK_SCENARIOS  — top-3 analyst scenarios for next 60 days")
print("  Context Agent  : Google ADK + google_search (gemini-3-flash-preview)")
print("  Analyst Agent  : litellm / gemini-3-flash-preview -> JSON")

Unified Analyst Agent loaded.
  System prompt  : _ANALYST_SYSTEM (shared across all tasks)
  Task A         : TASK_TRAJECTORY — point estimates at 5/10/21 bdays
  Task B         : TASK_SHOCK      — P(up > +$5) in 5 bdays
  Task C         : TASK_SCENARIOS  — top-3 analyst scenarios for next 60 days
  Context Agent  : Google ADK + google_search (gemini-3-flash-preview)
  Analyst Agent  : litellm / gemini-3-flash-preview -> JSON


---
## Stream 1 — Trajectory Forecast

**Question:** Where will WTI be in 5, 10, and 21 business days?

We run the full pipeline at **3 origins** that span Jan–Mar 2026:

| Origin | WTI at origin | What the news said |
|---|---|---|
| Jan 5, 2026 | ~\$60 | Quiet post-holiday market, mild demand concerns |
| Feb 2, 2026 | ~\$62 | Early geopolitical tension signals in the Gulf |
| Mar 2, 2026 | ~\$71 | Conflict onset — Hormuz closure announced |

Prophet extrapolates from history and predicts mean reversion in all three cases.  The Analyst Agent reads the news and can revise upward when conditions warrant it.

In [5]:
# ── Run or load trajectory agent forecasts ───────────────────────────────────
if TRAJ_AGENT_CACHE.exists() and TRAJ_CONTEXT_CACHE.exists():
    with open(TRAJ_AGENT_CACHE) as f:
        traj_agent_results: list[dict] = json.load(f)
    with open(TRAJ_CONTEXT_CACHE) as f:
        traj_contexts: dict[str, str] = json.load(f)
    print(f"Loaded {len(traj_agent_results)} cached trajectory forecasts.")
else:
    traj_contexts = {}
    traj_agent_results = []

    for origin in TRAJECTORY_ORIGINS:
        key = origin.strftime("%Y-%m-%d")
        cutoff = (origin - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])

        print(f"\n{'─' * 55}")
        print(f"Origin {key}  (cutoff: {cutoff})  WTI ${origin_price:.2f}")

        print("  [1/2] Context Agent ...")
        ctx = await retrieve_context(cutoff)  # noqa: F704, PLE1142
        traj_contexts[key] = ctx
        print(f"        {len(ctx):,} chars")

        print("  [2/2] Analyst Agent — Task A (Trajectory) ...")
        hist = compress_history(price_df, origin)
        hist_str = serialize_history(hist, precision=2)
        result = run_analyst(TASK_TRAJECTORY, hist_str, ctx, key, origin_price)
        result["origin"] = key
        traj_agent_results.append(result)
        print(
            f"        day_5={result.get('day_5', '?')}  day_10={result.get('day_10', '?')}  day_21={result.get('day_21', '?')}  conf={result.get('confidence', '?')}"
        )

    with open(TRAJ_AGENT_CACHE, "w") as f:
        json.dump(traj_agent_results, f, indent=2)
    with open(TRAJ_CONTEXT_CACHE, "w") as f:
        json.dump(traj_contexts, f, indent=2)
    print("\nSaved to cache.")

traj_agent_by_origin = {r["origin"]: r for r in traj_agent_results}
print("\nAgent trajectory summary:")
for r in traj_agent_results:
    key = r["origin"]
    origin_price = float(price_df[price_df.index >= pd.Timestamp(key)].iloc[0]["price"])
    print(
        f"  {key}  WTI=${origin_price:.2f}  "
        f"day_5={r.get('day_5', float('nan')):.1f}  "
        f"day_10={r.get('day_10', float('nan')):.1f}  "
        f"day_21={r.get('day_21', float('nan')):.1f}  "
        f"conf={r.get('confidence', '?')}"
    )

Loaded 3 cached trajectory forecasts.

Agent trajectory summary:
  2026-01-05  WTI=$58.32  day_5=60.5  day_10=61.8  day_21=59.6  conf=medium
  2026-02-02  WTI=$62.14  day_5=62.8  day_10=63.5  day_21=65.0  conf=medium
  2026-03-02  WTI=$71.23  day_5=82.5  day_10=91.0  day_21=96.0  conf=medium


In [6]:
# ── I/O inspection: Mar-2 — conflict onset, most informative ─────────────────
INSPECT_ORIGIN = "2026-03-02"
inspect_result = traj_agent_by_origin[INSPECT_ORIGIN]
inspect_ctx = traj_contexts[INSPECT_ORIGIN]
inspect_price = float(price_df[price_df.index >= pd.Timestamp(INSPECT_ORIGIN)].iloc[0]["price"])
hist = compress_history(price_df, pd.Timestamp(INSPECT_ORIGIN))
hist_lines = serialize_history(hist, precision=2).split("\n")

display(
    MD(
        "### Task A — I/O Inspection: " + INSPECT_ORIGIN + "\n\n"
        "Full input → output trace. The system prompt is **identical** for every task; "
        "only the TASK SPECIFICATION block in the user message changes."
    )
)

# ── Shared system prompt ──────────────────────────────────────────────────────
sys_preview = _ANALYST_SYSTEM.replace("\n", "\n> ")
display(
    MD(
        "<details><summary>📋 <strong>System prompt</strong> — shared across all tasks</summary>\n\n"
        "> " + sys_preview + "\n\n</details>"
    )
)

# ── User message ──────────────────────────────────────────────────────────────
ctx_preview = inspect_ctx[:600].strip() + " ..."
price_preview = "\n".join(hist_lines[-10:])
task_preview = TASK_TRAJECTORY.replace("\n", "\n> ")
user_display = (
    f"**Price history** — last 10 of {len(hist_lines)} rows:\n\n"
    "```\n" + price_preview + "\n```\n\n"
    f"**Market briefing** (first 600 chars, cutoff: {(pd.Timestamp(INSPECT_ORIGIN) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}):\n\n"
    + ctx_preview
    + "\n\n"
    "**TASK SPECIFICATION** (Task A — Trajectory):\n\n"
    "> " + task_preview
)
display(
    MD(
        "<details><summary>📤 <strong>User message</strong> (abbreviated)</summary>\n\n"
        + user_display
        + "\n\n</details>"
    )
)

# ── Parsed response ───────────────────────────────────────────────────────────
bday_dates = pd.bdate_range(start=pd.Timestamp(INSPECT_ORIGIN) + pd.offsets.BDay(1), periods=21)
actuals = {
    5: float(price_df[price_df.index >= bday_dates[4]].iloc[0]["price"]),
    10: float(price_df[price_df.index >= bday_dates[9]].iloc[0]["price"]),
    21: float(price_df[price_df.index >= bday_dates[20]].iloc[0]["price"]),
}

table_md = (
    "#### 📥 Agent response — Task A\n\n"
    "| Horizon | Agent estimate | 80% CI | Actual | Agent error | Prophet error |\n"
    "|---|---|---|---|---|---|\n"
)
for h_val, actual in actuals.items():
    p_row = prophet_traj_df[
        (prophet_traj_df["origin"] == pd.Timestamp(INSPECT_ORIGIN)) & (prophet_traj_df["horizon"] == h_val)
    ]
    p_yhat = float(p_row.iloc[0]["yhat"]) if not p_row.empty else float("nan")
    a_val = inspect_result.get(f"day_{h_val}", float("nan"))
    lo = inspect_result.get(f"lower_80_day_{h_val}", float("nan"))
    hi = inspect_result.get(f"upper_80_day_{h_val}", float("nan"))
    table_md += (
        f"| {h_val} bdays | **${a_val:.1f}** | [{lo:.1f} – {hi:.1f}] | ${actual:.1f} | "
        f"{a_val - actual:+.1f} | {p_yhat - actual:+.1f} |\n"
    )
table_md += (
    f"\n> **Reasoning:** {inspect_result.get('reasoning', 'N/A')}\n\n"
    f"**Confidence:** {(inspect_result.get('confidence') or '?').title()}"
)
display(MD(table_md))

### Task A — I/O Inspection: 2026-03-02

Full input → output trace. The system prompt is **identical** for every task; only the TASK SPECIFICATION block in the user message changes.

<details><summary>📋 <strong>System prompt</strong> — shared across all tasks</summary>

> You are an expert oil market analyst.
> 
> You will receive:
>   1. WTI crude oil price history (daily, most recent last)
>   2. An oil market intelligence briefing with a strict temporal cutoff
>   3. A TASK SPECIFICATION that defines the exact question and the required
>      JSON output schema
> 
> Read the data and briefing carefully, then execute the task precisely.
> Return ONLY valid JSON that matches the schema described in the task specification.
> Output ONLY the JSON object — no preamble, no explanation outside the JSON.

</details>

<details><summary>📤 <strong>User message</strong> (abbreviated)</summary>

**Price history** — last 10 of 368 rows:

```
2026-02-17 $62.33
2026-02-18 $65.19
2026-02-19 $66.43
2026-02-20 $66.39
2026-02-23 $66.31
2026-02-24 $65.63
2026-02-25 $65.42
2026-02-26 $65.21
2026-02-27 $67.02
2026-03-02 $71.23
```

**Market briefing** (first 600 chars, cutoff: 2026-03-01):

### **Oil Market Intelligence Briefing: March 1, 2026**

**Price Action and Immediate Trend**
As of March 1, 2026, the global oil market is entering a state of extreme volatility following the dramatic escalation of hostilities in the Persian Gulf over the last 24 hours. Prior to the weekend, WTI was trending modestly upward, closing on February 27 at approximately **$65.99**, with Brent near **$71.58**. However, the launch of "Operation Epic Fury" by U.S. and Israeli forces on February 28—which reportedly targeted Iranian military and nuclear infrastructure—has triggered a massive "black swan ...

**TASK SPECIFICATION** (Task A — Trajectory):

> Forecast the WTI crude oil price at three forward horizons from today:
>   - 5  business days (~1 trading week)
>   - 10 business days (~2 trading weeks)
>   - 21 business days (~1 calendar month)
> 
> For each horizon provide a point estimate and an 80% confidence interval.
> Be calibrated: historical weekly vol is roughly $2-5/bbl; widen intervals
> when the market is unusually uncertain.
> 
> Return JSON with exactly these fields:
> {
>   "day_5":           <float>,
>   "lower_80_day_5":  <float>,
>   "upper_80_day_5":  <float>,
>   "day_10":          <float>,
>   "lower_80_day_10": <float>,
>   "upper_80_day_10": <float>,
>   "day_21":          <float>,
>   "lower_80_day_21": <float>,
>   "upper_80_day_21": <float>,
>   "reasoning":       "<2-4 sentences>",
>   "confidence":      "<high|medium|low>"
> }

</details>

#### 📥 Agent response — Task A

| Horizon | Agent estimate | 80% CI | Actual | Agent error | Prophet error |
|---|---|---|---|---|---|
| 5 bdays | **$82.5** | [76.0 – 89.0] | $94.8 | -12.3 | -31.7 |
| 10 bdays | **$91.0** | [81.0 – 101.0] | $93.5 | -2.5 | -31.2 |
| 21 bdays | **$96.0** | [82.0 – 110.0] | $101.4 | -5.4 | -40.1 |

> **Reasoning:** The de facto closure of the Strait of Hormuz following Operation Epic Fury has triggered a massive geopolitical risk premium, removing approximately 20% of global supply. While U.S. SPR releases may provide a temporary buffer, the immediate supply vacuum and potential for infrastructure damage will drive WTI toward the $90-$100 range. Extreme volatility is expected as the market prices in the duration of the blockade and the risk of further escalation.

**Confidence:** Medium

In [7]:
# ── Trajectory comparison: Prophet fan + Agent point estimates ────────────
# Three panels, one per origin.  Prophet shows its full 95% CI fan.
# The Agent's discrete estimates at day 5/10/21 are plotted as markers
# with 80% CI error bars — a deliberately simple but honest representation.

HISTORY_WINDOW = 40  # business days of history to show before origin

fig = psp.make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        f"{o.strftime('%b %d')}: WTI ${float(price_df[price_df.index >= o].iloc[0]['price']):.0f}"
        for o in TRAJECTORY_ORIGINS
    ],
    shared_yaxes=True,
)

for col, origin in enumerate(TRAJECTORY_ORIGINS, start=1):
    key = origin.strftime("%Y-%m-%d")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    bday_dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=21)

    # History window
    hist_window = price_df[price_df.index <= origin].iloc[-HISTORY_WINDOW:]
    fig.add_trace(
        go.Scatter(
            x=hist_window.index,
            y=hist_window["price"],
            line={"color": CLR_ACTUAL, "width": 1.5},
            name="Actual",
            showlegend=(col == 1),
            legendgroup="actual",
        ),
        row=1,
        col=col,
    )

    # Actual post-origin prices
    actual_future = price_df[(price_df.index > origin) & (price_df.index <= bday_dates[-1])]
    fig.add_trace(
        go.Scatter(
            x=actual_future.index,
            y=actual_future["price"],
            line={"color": CLR_ACTUAL, "width": 2.5},
            name="Actual (outcome)",
            showlegend=(col == 1),
            legendgroup="actual",
        ),
        row=1,
        col=col,
    )

    # Prophet fan (95% CI)
    p_sub = prophet_traj_df[prophet_traj_df["origin"] == origin].sort_values("horizon")
    fig.add_trace(
        go.Scatter(
            x=pd.concat([p_sub["forecast_date"], p_sub["forecast_date"][::-1]]),
            y=pd.concat([p_sub["yhat_lower"], p_sub["yhat_upper"][::-1]]),
            fill="toself",
            fillcolor=CLR_PROPHET,
            opacity=0.15,
            line={"width": 0},
            showlegend=False,
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=p_sub["forecast_date"],
            y=p_sub["yhat"],
            line={"color": CLR_PROPHET, "width": 2, "dash": "dot"},
            name="Prophet",
            showlegend=(col == 1),
            legendgroup="prophet",
        ),
        row=1,
        col=col,
    )

    # Agent point estimates + 80% CI error bars
    a = traj_agent_by_origin.get(key, {})
    agent_horizons = [5, 10, 21]
    agent_dates = [bday_dates[h - 1] for h in agent_horizons]
    agent_meds = [a.get(f"day_{h}", float("nan")) for h in agent_horizons]
    agent_lo = [a.get(f"lower_80_day_{h}", float("nan")) for h in agent_horizons]
    agent_hi = [a.get(f"upper_80_day_{h}", float("nan")) for h in agent_horizons]
    err_hi = [hi - med if not (np.isnan(hi) or np.isnan(med)) else 0 for hi, med in zip(agent_hi, agent_meds)]
    err_lo = [med - lo if not (np.isnan(lo) or np.isnan(med)) else 0 for lo, med in zip(agent_lo, agent_meds)]
    fig.add_trace(
        go.Scatter(
            x=agent_dates,
            y=agent_meds,
            mode="markers",
            marker={"color": CLR_AGENT, "size": 11, "symbol": "diamond"},
            error_y={
                "type": "data",
                "symmetric": False,
                "array": err_hi,
                "arrayminus": err_lo,
                "color": CLR_AGENT,
                "thickness": 2,
                "width": 6,
            },
            name="Agent",
            showlegend=(col == 1),
            legendgroup="agent",
        ),
        row=1,
        col=col,
    )

    # Origin marker
    fig.add_vline(
        x=origin.timestamp() * 1000,
        line={"color": "#aaa", "dash": "dash", "width": 1},
        row=1,
        col=col,
    )

fig.update_layout(
    title={"text": "WTI Trajectory — Prophet (fan) vs. Analyst Agent (markers)", "x": 0.5, "font": {"size": 13}},
    height=380,
    width=900,
    template="plotly_white",
    legend={"orientation": "h", "y": -0.20, "x": 0.5, "xanchor": "center"},
    margin={"t": 70, "b": 80, "l": 55, "r": 20},
)
fig.update_yaxes(title_text="WTI (USD/bbl)", col=1)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#ececec", gridwidth=0.7)
fig.show()

In [8]:
# ── Trajectory evaluation: MAE at 3 horizons ─────────────────────────────
eval_rows = []
for origin in TRAJECTORY_ORIGINS:
    key = origin.strftime("%Y-%m-%d")
    bday_dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=21)
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    a = traj_agent_by_origin.get(key, {})

    for h in [5, 10, 21]:
        actual_date = bday_dates[h - 1]
        actual = float(price_df[price_df.index >= actual_date].iloc[0]["price"])
        p_row = prophet_traj_df[(prophet_traj_df["origin"] == origin) & (prophet_traj_df["horizon"] == h)]
        p_yhat = float(p_row.iloc[0]["yhat"]) if not p_row.empty else float("nan")
        a_yhat = a.get(f"day_{h}", float("nan"))
        eval_rows.append(
            {
                "Origin": key,
                "Horizon": f"{h} bdays",
                "Actual ($)": f"{actual:.1f}",
                "Prophet ($)": f"{p_yhat:.1f}",
                "Agent ($)": f"{a_yhat:.1f}",
                "Prophet MAE": abs(p_yhat - actual),
                "Agent MAE": abs(a_yhat - actual),
            }
        )

eval_df = pd.DataFrame(eval_rows)
# Mean MAE across all origins × horizons
mean_mae = eval_df[["Prophet MAE", "Agent MAE"]].mean()
display_df = eval_df.drop(columns=["Prophet MAE", "Agent MAE"]).set_index(["Origin", "Horizon"])
display(display_df)
print(f"\nMean MAE  Prophet: ${mean_mae['Prophet MAE']:.2f}  Agent: ${mean_mae['Agent MAE']:.2f}")

Actual ($) Prophet ($) Agent ($)
Origin     Horizon                                  
2026-01-05 5 bdays        59.5        56.9      60.5
           10 bdays       60.3        57.6      61.8
           21 bdays       63.2        57.6      59.6
2026-02-02 5 bdays        64.4        58.3      62.8
           10 bdays       62.3        58.7      63.5
           21 bdays       74.6        60.9      65.0
2026-03-02 5 bdays        94.8        63.1      82.5
           10 bdays       93.5        62.3      91.0
           21 bdays      101.4        61.3      96.0


Mean MAE  Prophet: $15.25  Agent: $4.27


---
## Stream 2 — Binary Shock Prediction

**Question:** Will WTI close more than \$5/bbl *higher* than today within 5 trading days?

We ask this question weekly from **Feb 2 through Mar 23, 2026** — 8 origins spanning the calm pre-conflict period and the volatile post-conflict weeks.  This gives us 8 independent test cases with known outcomes.

**Why this question exposes Prophet's weakness:**  Prophet's mean-reversion model assigns near-zero probability to a further \$5+ upside move when prices are already elevated (\$71–\$95).  But conflict-driven rallies don't mean-revert on a weekly basis.  An agent that reads the news about a Hormuz closure can assign 80%+ — and be right.

| Origin | WTI | 5-day Δ | Outcome |
|---|---|---|---|
| Feb 2 | \$62.14 | +\$2.22 | No shock |
| Feb 9 | \$64.36 | −\$2.03 | No shock |
| Feb 17 | \$62.33 | +\$3.30 | No shock |
| Feb 23 | \$66.31 | +\$4.92 | No shock |
| **Mar 2** | \$71.23 | **+\$23.54** | **Shock** |
| Mar 9 | \$94.77 | −\$1.27 | No shock |
| Mar 16 | \$93.50 | −\$5.37 | No shock |
| **Mar 23** | \$88.13 | **+\$14.75** | **Shock** |


In [9]:
# ── Run or load binary shock forecasts ───────────────────────────────────────
if SHOCK_ANALYST_CACHE.exists() and SHOCK_CONTEXT_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_analyst_results: list[dict] = json.load(f)
    with open(SHOCK_CONTEXT_CACHE) as f:
        shock_news_contexts: dict[str, str] = json.load(f)
    print(f"Loaded {len(shock_analyst_results)} cached shock forecasts.")
else:
    shock_news_contexts = {}
    shock_analyst_results = []

    for origin in SHOCK_ORIGINS:
        key = origin.strftime("%Y-%m-%d")
        cutoff = (origin - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])

        print(f"\n{'─' * 55}")
        print(f"Origin {key}  (cutoff: {cutoff})  WTI ${origin_price:.2f}")

        print("  [1/2] Context Agent ...")
        ctx = await retrieve_context(cutoff)  # noqa: F704, PLE1142
        shock_news_contexts[key] = ctx
        print(f"        {len(ctx):,} chars")

        print("  [2/2] Analyst Agent — Task B (Shock) ...")
        hist = compress_history(price_df, origin)
        hist_str = serialize_history(hist, precision=2)
        result = run_analyst(TASK_SHOCK, hist_str, ctx, key, origin_price)
        result["origin"] = key
        shock_analyst_results.append(result)
        p_up = result.get("probability_up", float("nan"))
        print(f"        P(up>+${SHOCK_THRESHOLD:.0f})={p_up:.2f}  conf={result.get('confidence', '?')}")

    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_analyst_results, f, indent=2)
    with open(SHOCK_CONTEXT_CACHE, "w") as f:
        json.dump(shock_news_contexts, f, indent=2)
    print("\nSaved to cache.")

shock_analyst_by_origin = {r["origin"]: r for r in shock_analyst_results}

# ── Assemble binary_df ────────────────────────────────────────────────────────
binary_rows = []
for origin in SHOCK_ORIGINS:
    key = origin.strftime("%Y-%m-%d")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, delta = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)
    pt_sub = prophet_shock_traj_df[prophet_shock_traj_df["origin"] == origin]
    p_prob = prophet_prob_shock(pt_sub, origin_price, SHOCK_THRESHOLD)
    a = shock_analyst_by_origin.get(key, {})
    a_prob = float(a.get("probability_up", float("nan")))
    for method, prob in [("Prophet", p_prob), ("Analyst Agent", a_prob)]:
        binary_rows.append(
            {
                "origin": key,
                "origin_price": origin_price,
                "delta": delta,
                "outcome": outcome,
                "method": method,
                "prob": prob,
                "brier": (prob - outcome) ** 2,
                "reasoning": a.get("reasoning") if method == "Analyst Agent" else None,
                "key_signals": a.get("key_signals", []) if method == "Analyst Agent" else [],
                "confidence": a.get("confidence") if method == "Analyst Agent" else None,
                "direction_bias": a.get("direction_bias") if method == "Analyst Agent" else None,
            }
        )

binary_df = pd.DataFrame(binary_rows)
binary_df["origin_dt"] = pd.to_datetime(binary_df["origin"])
binary_df = binary_df.sort_values(["origin_dt", "method"]).reset_index(drop=True)
print(f"binary_df: {len(binary_df)} rows  ({binary_df['origin'].nunique()} origins x 2 methods)")

Loaded 8 cached shock forecasts.
binary_df: 16 rows  (8 origins x 2 methods)


In [10]:
# ── I/O inspection: Mar-2 (clearest correct call) ────────────────────────────
INSPECT_SHOCK_ORIGIN = "2026-03-02"
s_result = shock_analyst_by_origin[INSPECT_SHOCK_ORIGIN]
s_ctx = shock_news_contexts[INSPECT_SHOCK_ORIGIN]
s_price = float(price_df[price_df.index >= pd.Timestamp(INSPECT_SHOCK_ORIGIN)].iloc[0]["price"])
outcome, delta = check_shock_outcome(price_df, pd.Timestamp(INSPECT_SHOCK_ORIGIN), SHOCK_THRESHOLD, SHOCK_HORIZON)
hist = compress_history(price_df, pd.Timestamp(INSPECT_SHOCK_ORIGIN))
hist_lines = serialize_history(hist, precision=2).split("\n")

display(
    MD(
        "### Task B — I/O Inspection: " + INSPECT_SHOCK_ORIGIN + "\n\n"
        "Same agent, same system prompt — only the TASK SPECIFICATION changes."
    )
)

# ── Shared system prompt (same as Task A) ─────────────────────────────────────
sys_preview = _ANALYST_SYSTEM.replace("\n", "\n> ")
display(
    MD(
        "<details><summary>📋 <strong>System prompt</strong> — same as Task A</summary>\n\n"
        "> " + sys_preview + "\n\n</details>"
    )
)

# ── User message ──────────────────────────────────────────────────────────────
ctx_preview = s_ctx[:600].strip() + " ..."
price_preview = "\n".join(hist_lines[-10:])
task_preview = TASK_SHOCK.replace("\n", "\n> ")
user_display = (
    f"**Price history** — last 10 of {len(hist_lines)} rows:\n\n"
    "```\n" + price_preview + "\n```\n\n"
    f"**Market briefing** (first 600 chars, cutoff: {(pd.Timestamp(INSPECT_SHOCK_ORIGIN) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}):\n\n"
    + ctx_preview
    + "\n\n"
    "**TASK SPECIFICATION** (Task B — Binary Shock):\n\n"
    "> " + task_preview
)
display(
    MD(
        "<details><summary>📤 <strong>User message</strong> (abbreviated)</summary>\n\n"
        + user_display
        + "\n\n</details>"
    )
)

# ── Parsed response + verdict ─────────────────────────────────────────────────
a_prob = float(s_result.get("probability_up", float("nan")))
brier = (a_prob - outcome) ** 2
signals = " · ".join(s_result.get("key_signals", [])) or "—"

display(
    MD(
        "#### 📥 Agent response — Task B\n\n"
        f"| | |\n|---|---|\n"
        f"| **P(up > +${SHOCK_THRESHOLD:.0f})** | **{a_prob:.0%}** |\n"
        f"| Direction bias | {s_result.get('direction_bias', '?').title()} |\n"
        f"| Confidence | {(s_result.get('confidence') or '?').title()} |\n"
        f"| Key signals | {signals} |\n\n"
        f"> **Reasoning:** {s_result.get('reasoning', 'N/A')}\n\n"
        "---\n\n"
        "#### Outcome\n\n"
        f"| | |\n|---|---|\n"
        f"| Actual 5-day move | **{delta:+.2f} /bbl** → {'🔴 SHOCK' if outcome else '🟢 No shock'} |\n"
        f"| Verdict | {'✅ Correct — predicted shock, shock happened' if (a_prob >= 0.5 and outcome) else '✅ Correct — predicted no shock' if (a_prob < 0.5 and not outcome) else '❌ Incorrect'} |\n"
        f"| Brier score | {brier:.3f} {'🟢 excellent' if brier < 0.05 else '🟢 good' if brier < 0.10 else '🟡 ok' if brier < 0.25 else '🔴 poor'} |\n"
    )
)

### Task B — I/O Inspection: 2026-03-02

Same agent, same system prompt — only the TASK SPECIFICATION changes.

<details><summary>📋 <strong>System prompt</strong> — same as Task A</summary>

> You are an expert oil market analyst.
> 
> You will receive:
>   1. WTI crude oil price history (daily, most recent last)
>   2. An oil market intelligence briefing with a strict temporal cutoff
>   3. A TASK SPECIFICATION that defines the exact question and the required
>      JSON output schema
> 
> Read the data and briefing carefully, then execute the task precisely.
> Return ONLY valid JSON that matches the schema described in the task specification.
> Output ONLY the JSON object — no preamble, no explanation outside the JSON.

</details>

<details><summary>📤 <strong>User message</strong> (abbreviated)</summary>

**Price history** — last 10 of 368 rows:

```
2026-02-17 $62.33
2026-02-18 $65.19
2026-02-19 $66.43
2026-02-20 $66.39
2026-02-23 $66.31
2026-02-24 $65.63
2026-02-25 $65.42
2026-02-26 $65.21
2026-02-27 $67.02
2026-03-02 $71.23
```

**Market briefing** (first 600 chars, cutoff: 2026-03-01):

### **Oil Market Intelligence Briefing: March 1, 2026**

**Immediate Catalyst: The "Black Saturday" Strikes and Geopolitical Escalation**
The primary driver for the coming week is the massive geopolitical shock following the February 28, 2026, coordinated air strikes by U.S. and Israeli forces against Iranian military and nuclear infrastructure. As of today, Sunday, March 1, global markets are bracing for a violent "gap up" at the Monday open. WTI, which closed Friday at approximately $66–$67/bbl, is expected to surge toward the $80–$85 range immediately. The reported assassination of Iran’s S ...

**TASK SPECIFICATION** (Task B — Binary Shock):

> Estimate P(up) — the probability that WTI will close MORE THAN
> $5/bbl HIGHER than today's price at the end of
> 5 trading days.
> 
> This is a directional upside question only.
> 
> Calibration guidance:
>   - No unusual upside catalyst       -> base rate ~10-15%
>   - Escalating unconfirmed risk      -> 20-40%
>   - Confirmed supply disruption      -> 60-85%
> 
> Return JSON with exactly these fields:
> {
>   "probability_up":  <float 0-1>,
>   "direction_bias":  "<up|down|neutral>",
>   "reasoning":       "<2-4 sentences>",
>   "key_signals":     ["<signal 1>", "<signal 2>", "<signal 3>"],
>   "confidence":      "<high|medium|low>"
> }

</details>

#### 📥 Agent response — Task B

| | |
|---|---|
| **P(up > +$5)** | **82%** |
| Direction bias | Up |
| Confidence | High |
| Key signals | Closure of the Strait of Hormuz · US-Israeli strikes on Iranian nuclear/military infrastructure · Suspension of Persian Gulf transit by major shipping conglomerates |

> **Reasoning:** The assassination of Iran's Supreme Leader and the subsequent blockade of the Strait of Hormuz represent a catastrophic geopolitical supply shock. With 20% of global oil supply effectively halted and major shipping lines suspending transit, the market is pricing in a massive war premium that historically leads to double-digit price spikes. A $5 move from the current $71.23 level is a conservative threshold given that analysts are already targeting the $80-$85 range in the immediate term.

---

#### Outcome

| | |
|---|---|
| Actual 5-day move | **+23.54 /bbl** → 🔴 SHOCK |
| Verdict | ✅ Correct — predicted shock, shock happened |
| Brier score | 0.032 🟢 excellent |


In [11]:
# ── One forecast card per origin ─────────────────────────────────────────
def _verdict(a_prob: float, outcome: int, delta: float, threshold: float) -> str:
    if outcome == 1:
        return (
            f"✅ **Correct** — predicted shock, price rose +${delta:.2f}"
            if a_prob >= 0.50
            else f"❌ **Missed** — predicted no shock, price rose +${delta:.2f}"
        )
    return (
        f"✅ **Correct** — predicted no shock, price moved {delta:+.2f}"
        if a_prob < 0.50
        else f"❌ **False alarm** — predicted shock, price moved {delta:+.2f}"
    )


def _bar(val: float, width: int = 10) -> str:
    n = int(round(val * width))
    return "█" * n + "░" * (width - n)


def _conf_bar(conf: str) -> str:
    return {"high": "████░", "medium": "███░░", "low": "██░░░"}.get((conf or "").lower(), "?????")


for origin in SHOCK_ORIGINS:
    key = origin.strftime("%Y-%m-%d")
    label = origin.strftime("%b %-d, %Y")
    a = shock_analyst_by_origin.get(key, {})
    ctx = shock_news_contexts.get(key, "*(no context)*")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, delta = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)
    a_prob = float(a.get("probability_up", float("nan")))
    confidence = a.get("confidence", "?")
    brier = (a_prob - outcome) ** 2
    cutoff = (origin - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    outcome_badge = "🔴 **SHOCK**" if outcome else "🟢 **No shock**"
    display(
        MD(f"""
---
### {label} — WTI ${origin_price:.2f} /bbl

| | |
|---|---|
| **Prediction** | P(up > +${SHOCK_THRESHOLD:.0f}) = **{a_prob:.0%}**  `{_bar(a_prob)}` |
| **Confidence** | {confidence.title() if confidence else "?"}  `{_conf_bar(confidence)}` |
| **Rationale** | {a.get("reasoning", "N/A")} |
| **Key signals** | {" · ".join(a.get("key_signals", [])) or "—"} |
| **Actual outcome** | {outcome_badge} — price moved **{delta:+.2f} /bbl** |
| **Verdict** | {_verdict(a_prob, outcome, delta, SHOCK_THRESHOLD)} |
| **Brier score** | {brier:.3f} {"🟢" if brier < 0.10 else "🟡" if brier < 0.25 else "🔴"} |

<details><summary>📰 Oil market context (cutoff: {cutoff})</summary>

{ctx}

</details>
""")
    )


---
### Feb 2, 2026 — WTI $62.14 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **12%**  `█░░░░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently weighed down by a significant projected global supply surplus of 2.1 to 4.0 million bpd, which creates a strong fundamental headwind. While the briefing identifies upside risks such as SPR refill announcements and geopolitical tensions, the recent sharp decline from $65 to $62.14 suggests bearish momentum is currently dominant. A $5 rally in five trading days would require an immediate and major catalyst to overcome the prevailing supply glut narrative. |
| **Key signals** | Projected 2026 global oil surplus · U.S. SPR buy-back policy rhetoric · Red Sea maritime security levels |
| **Actual outcome** | 🟢 **No shock** — price moved **+2.22 /bbl** |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved +2.22 |
| **Brier score** | 0.014 🟢 |

<details><summary>📰 Oil market context (cutoff: 2026-02-01)</summary>

### **Oil Market Intelligence Briefing: February 1, 2026**

**Price Levels and Market Sentiment**
As of February 1, 2026, WTI crude is trading in a bearish corridor between **$53 and $57 per barrel**, while Brent remains pressured near the **$60 mark**. The market is currently dominated by a "supply glut" narrative, with the EIA and IEA both forecasting a significant global surplus of **2.1 to 4.0 million barrels per day (bpd)** for the first half of 2026. This bearishness is driven by record-breaking non-OPEC+ production growth from the U.S., Brazil, and Guyana, which has consistently outpaced sluggish demand growth in the petrochemical and aviation sectors.

**OPEC+ Production Strategy**
In response to deteriorating market conditions, OPEC+ has implemented a **"strategic pause"** for the first quarter of 2026. The alliance, led by Saudi Arabia and Russia, recently confirmed it will freeze all planned production hikes through March to prevent a total price collapse. While this move was intended to provide a floor, internal tensions are rising; the UAE’s recent signals regarding its long-term production capacity and the ongoing "compensation" disputes with over-producers like Iraq and Kazakhstan have created a fragile unity. Analysts warn that if prices dip below $50 WTI, the group may be forced into an emergency meeting to consider deeper, involuntary cuts.

**Geopolitical Risks and Shipping Lanes**
The Persian Gulf and Red Sea remain in a state of **"fragile equilibrium."** While large-scale disruptions have been avoided in the opening weeks of 2026, maritime security remains at a "Critical" level. Houthi activity in the Red Sea continues to force roughly 50% of Suez-bound traffic to reroute around the Cape of Good Hope, maintaining a persistent "risk premium" in freight and insurance costs. Furthermore, the U.S. administration has recently ratcheted up rhetoric against Iran and Venezuela, with new warnings regarding airspace and maritime "contraband" checks. Any tactical miscalculation in the Strait of Hormuz—which still carries 20% of global supply—remains the primary "black swan" candidate for a sudden $10–$15 price spike.

**U.S. Policy and Strategic Petroleum Reserve (SPR)**
The U.S. Strategic Petroleum Reserve currently sits at approximately **411 million barrels**, its lowest level in decades. The Trump administration has declared refilling the SPR a "Department-level priority" for 2026, seeking to capitalize on the current price dip. With WTI trading well below the administration's unofficial **$75–$80 target range**, market participants are closely watching for a large-scale "buy-back" announcement. Such a move would provide a significant psychological floor and could trigger a sharp short-term rally as the government competes with commercial refiners for physical barrels.

**5–10 Day Volatility Outlook**
The risk of a **sudden large move** in the next 5–10 days is skewed to the upside. While the fundamental outlook is bearish, the market is "short-heavy," making it vulnerable to a short squeeze. Key triggers to watch include:
1.  **Persian Gulf Escalation:** Any confirmed strike on energy infrastructure or a "closure" threat in the Strait of Hormuz.
2.  **SPR Intervention:** A formal DOE solicitation to purchase 10M+ barrels for the reserve.
3.  **OPEC+ Rhetoric:** Surprise signals from Riyadh regarding an early end to the "strategic pause" in favor of deeper cuts.
Conversely, a break below the **$50 support level** for WTI could trigger automated sell-offs if the 2026 surplus projections are further revised upward in upcoming February agency reports.

</details>



---
### Feb 9, 2026 — WTI $64.36 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **22%**  `██░░░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently dominated by a bearish supply glut narrative, which has led to heavy short positioning that is vulnerable to a squeeze. While the 'Trump Floor' at $60/bbl provides support, a $5 move in five days likely requires a kinetic geopolitical catalyst in the Persian Gulf, where tensions are currently simmering but unconfirmed. Given the 15% rise in maritime insurance premiums and recent drone sightings, the tail risk of a sudden spike is elevated above base rates. |
| **Key signals** | Persian Gulf insurance premium trends · SPR solicitation volumes · OPEC+ internal cohesion rumors |
| **Actual outcome** | 🟢 **No shock** — price moved **-2.03 /bbl** |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved -2.03 |
| **Brier score** | 0.048 🟢 |

<details><summary>📰 Oil market context (cutoff: 2026-02-08)</summary>

### **Oil Market Intelligence Briefing**
**Date:** February 8, 2026
**Subject:** WTI Crude Outlook and Supply Risk Assessment

#### **Current Price Levels and Market Sentiment**
As of early February 2026, **WTI crude is trading in the $62–$65/bbl range**, with Brent hovering near $68/bbl. The prevailing market sentiment is characterized by a "bearish overhang" following the International Energy Agency’s (IEA) January 21 report, which warned of a looming **global supply glut** that could peak at 4.5 million b/d by Q2 2026. This surplus is driven primarily by surging production from the "Americas Quintet" (U.S., Canada, Brazil, Guyana, and Argentina). However, prices have found a firm floor due to the Trump administration’s aggressive mandate to refill the **U.S. Strategic Petroleum Reserve (SPR)**, which currently stands at approximately 415 million barrels. The Department of Energy (DOE) is actively purchasing 1–2 million barrels per month, signaling a "Trump Floor" near $60/bbl.

#### **OPEC+ Policy and Supply Outlook**
OPEC+ is currently in a **production pause for Q1 2026**, having reaffirmed in late 2025 that they would suspend planned output increases for January, February, and March to counter seasonal demand weakness. While the group is officially keeping 3.6 million b/d off the market through year-end, internal cohesion is under scrutiny. Rumors regarding a potential **UAE exit from OPEC** have intensified, following the UAE’s successful bid for a higher production baseline. Market participants are also closely monitoring "compensation cuts" from Iraq and Kazakhstan, who have historically struggled with compliance. Any signal that OPEC+ might abandon its Q1 pause early to reclaim market share from U.S. shale would likely trigger a sharp downward move in WTI toward the $55 level.

#### **Geopolitical Risks and Shipping Disruptions**
While the market is currently focused on oversupply, **geopolitical "tail risks" in the Persian Gulf** are at their highest levels in years. Tensions between the U.S./Israel and Iran are described by analysts as "simmering at a crossroads." Although the Strait of Hormuz remains open as of today, insurance premiums for VLCCs (Very Large Crude Carriers) transiting the Gulf have risen by 15% over the last fortnight due to a series of "shadow war" incidents and drone sightings near regional energy infrastructure. The market is currently pricing in a very low probability of a full blockade; however, with 21 million b/d (20% of global supply) passing through this chokepoint, the disconnect between "glut pricing" and "conflict risk" is extreme.

#### **5–10 Day Outlook: Triggers for a Sudden Move**
The potential for a **sudden $5–$10 spike in WTI** over the next 10 days is high, primarily due to the market's heavy short positioning on the "supply glut" narrative. Key triggers to watch include:
*   **Maritime Incidents:** Any kinetic strike or seizure of a tanker in the Strait of Hormuz would cause an immediate short squeeze, as the IEA’s projected surplus cannot compensate for a disruption of that magnitude.
*   **U.S. Policy Shifts:** Unexpected signals regarding new tariffs on energy-exporting partners could dampen demand expectations further, while a sudden acceleration in SPR solicitations would tighten the physical prompt market.
*   **OPEC+ Rhetoric:** Ahead of the March 2026 window, any "leaks" regarding the extension of the production pause into Q2 would be viewed as a bullish signal, potentially lifting WTI back toward $70/bbl.

</details>



---
### Feb 17, 2026 — WTI $62.33 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **18%**  `██░░░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | A $5 increase in five trading days represents an approximate 8% rally, which is historically rare without a confirmed supply shock or major geopolitical escalation. While tightened sanctions on Russia and Persian Gulf tensions provide upside risk, the market is currently dominated by bearish IEA projections of a 4.5 million b/d surplus and seasonal demand weakness. The 'Trump Floor' created by SPR buying at $60-$63 provides strong support but does not inherently act as a catalyst for a sharp upward spike. |
| **Key signals** | IEA projected Q2 supply glut · OPEC+ March meeting rumors · Russian sanction escalation headlines |
| **Actual outcome** | 🟢 **No shock** — price moved **+3.30 /bbl** |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved +3.30 |
| **Brier score** | 0.032 🟢 |

<details><summary>📰 Oil market context (cutoff: 2026-02-16)</summary>

As of February 16, 2026, the crude oil market is characterized by a fragile stability, with WTI trading in the **$62–$65 range**. While prices have recovered from the five-month lows of $60 seen in late October 2025, the market remains under pressure from a looming "supply glut" projected for the second quarter of 2026. Sentiment is currently balanced between OPEC+ production discipline and a surge in output from the "Americas quintet" (US, Canada, Brazil, Guyana, and Argentina), which the IEA recently warned could lead to a massive 4.5 million b/d surplus by mid-year.

**OPEC+ Policy and Supply Outlook**
OPEC+ has taken a "proactively defensive" stance to start the year. Following their January 4, 2026, virtual meeting, the group reaffirmed a **pause in production increases** for February and March. This decision was driven by seasonal demand weakness and the need to offset the 2.9 million b/d added to the market in the latter half of 2025. Key members, including Iraq and the UAE, are currently under a strict "compensation schedule" to rectify previous overproduction, a move intended to signal group unity despite internal pressure from the UAE to increase its baseline capacity.

**Geopolitical Risks and Shipping**
Geopolitical risk premiums are currently centered on the friction surrounding **tightened sanctions on Russian oil majors** (Rosneft and Lukoil) and ongoing "wobbles" in the Persian Gulf. While the Strait of Hormuz remains open and functional, market participants are closely monitoring the "trade talks" in Washington, which have introduced volatility regarding the future of energy-related tariffs and sanctions. Any sudden breakdown in these diplomatic channels or a retaliatory move by Russia against the latest US/UK sanctions could trigger a rapid $5–$10 spike in WTI as shorts cover.

**US Energy Policy and the SPR**
Under the current administration, the **Strategic Petroleum Reserve (SPR)** has shifted from a source of supply to a significant source of demand. As of mid-February 2026, the SPR stands at approximately **415 million barrels**, up from 395 million a year ago. The Department of Energy (DOE) has made refilling the reserve a "national security priority," with a standing strategy to purchase crude whenever WTI dips toward the $60–$63 level. This "Trump Floor" is providing a critical psychological support level for WTI, preventing a deeper slide despite bearish IEA demand forecasts.

**5–10 Day Volatility Outlook**
Over the next 5–10 days, WTI is at risk of a **sudden large move** triggered by two primary factors:
1.  **Downside Risk:** If upcoming weekly inventory data confirms the IEA’s "bloated inventory" thesis or if Chinese demand indicators for Q1 underperform, WTI could test the $60 support level.
2.  **Upside Risk:** The market is highly sensitive to "headline risk" regarding the Persian Gulf. With several OPEC+ members meeting again on March 1, any rumors of an extended pause or deeper voluntary cuts to combat the Q2 glut could spark a sharp relief rally toward $70. Analysts at Goldman Sachs and JPMorgan have warned that while fundamentals are "structurally weak," the low level of OECD commercial inventories makes the market prone to extreme volatility on any physical supply disruption.

</details>



---
### Feb 23, 2026 — WTI $66.31 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **38%**  `████░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently pricing in a significant geopolitical risk premium due to imminent threats of military escalation in the Persian Gulf and potential disruptions to the Strait of Hormuz. While physical fundamentals suggest a surplus, the intelligence briefing indicates a high probability of a supply shock within the 5-10 day window. A $5 move is well within the range of volatility expected if kinetic action or major shipping incidents occur. |
| **Key signals** | Military posturing in the Persian Gulf · Strait of Hormuz shipping notices · OPEC+ production cut extensions |
| **Actual outcome** | 🟢 **No shock** — price moved **+4.92 /bbl** |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved +4.92 |
| **Brier score** | 0.144 🟡 |

<details><summary>📰 Oil market context (cutoff: 2026-02-22)</summary>

### **Oil Market Intelligence Briefing**
**Date:** February 22, 2026  
**Subject:** Supply Risks and Geopolitical Volatility Outlook (5–10 Day Window)

#### **Price Trend and Market Sentiment**
As of late February 2026, WTI crude is trading in the **$68–$71 range**, showing a distinct bullish tilt after a period of relative stability. While the IEA’s February 12 report highlighted a supply-demand surplus in early Q1 due to record non-OPEC+ production in 2025, prices have recently decoupled from these fundamentals. A "geopolitical risk premium" of approximately $8–$10 has begun to embed into Brent and WTI benchmarks over the last 10 days. This shift is driven by intensifying rumors of a shift in U.S. and Israeli posture toward Iran, moving the market from a "bearish surplus" narrative to one of "imminent supply shock" anticipation.

#### **OPEC+ Policy and Supply Outlook**
The OPEC+ alliance remains the primary floor for prices, having solidified its strategy in late 2025. On December 5, 2025, the group formally **extended its current production cuts through December 31, 2026**. This includes the 1.65 million bpd voluntary adjustments originally announced in 2023. While the group had initially planned to phase out some cuts in early 2026, the January 2026 ministerial monitoring committee (JMMC) signaled a "cautious and flexible" approach, effectively pausing any production increases. This discipline has successfully offset the 1.2 million bpd supply plunge seen in January caused by severe winter weather disruptions in North America and outages in Kazakhstan.

#### **Geopolitical Risks and the Persian Gulf**
The most critical factor for the next 5–10 days is the **deteriorating security situation in the Persian Gulf**. Intelligence signals and diplomatic rhetoric from Washington and Jerusalem suggest that the "shadow war" with Tehran is reaching a breaking point. Market participants are pricing in a high probability of a direct military escalation before the end of the month. The primary concern is a de facto closure or significant disruption of the **Strait of Hormuz**, which handles roughly 20% of global oil consumption. Any kinetic action against Iranian infrastructure or a retaliatory blockade by the IRGC would immediately strand 11–16 million bpd of Gulf exports, a disruption for which there is no global equivalent in spare capacity.

#### **US Strategic Petroleum Reserve (SPR) and Policy**
The U.S. energy policy under the current administration has shifted aggressively toward **refilling the Strategic Petroleum Reserve**. As of the week ending February 13, 2026, the SPR stands at **415.4 million barrels**, up approximately 15% from 2024 levels. While the Department of Energy (DOE) continues to award small-scale refill contracts (most recently for 1 million barrels at the Bryan Mound site), the administration has signaled that the SPR is now being held as a "strategic weapon" rather than a price-control tool. This suggests that the U.S. may be less willing to release emergency barrels to dampen minor price spikes, preferring to keep the reserve as a buffer against the looming Middle Eastern conflict.

#### **5–10 Day Forecast: Sudden Move Triggers**
The potential for a **sudden $20–$30 move in WTI** over the next 5–10 days is at its highest level in years. The "trigger" is expected to be a military strike or a major shipping incident in the Strait of Hormuz. If hostilities commence, analysts project WTI could breach **$100/bbl** almost instantly, as global inventories—despite recent builds—cannot compensate for a total Persian Gulf shut-in. Conversely, if a diplomatic "de-escalation" occurs, WTI would likely collapse back toward its fundamental support at **$60/bbl**. Traders should watch for any "Notice to Mariners" in the Gulf or sudden shifts in U.S. carrier strike group positions as the primary lead indicators for this move.

</details>



---
### Mar 2, 2026 — WTI $71.23 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **82%**  `████████░░` |
| **Confidence** | High  `████░` |
| **Rationale** | The assassination of Iran's Supreme Leader and the subsequent blockade of the Strait of Hormuz represent a catastrophic geopolitical supply shock. With 20% of global oil supply effectively halted and major shipping lines suspending transit, the market is pricing in a massive war premium that historically leads to double-digit price spikes. A $5 move from the current $71.23 level is a conservative threshold given that analysts are already targeting the $80-$85 range in the immediate term. |
| **Key signals** | Closure of the Strait of Hormuz · US-Israeli strikes on Iranian nuclear/military infrastructure · Suspension of Persian Gulf transit by major shipping conglomerates |
| **Actual outcome** | 🔴 **SHOCK** — price moved **+23.54 /bbl** |
| **Verdict** | ✅ **Correct** — predicted shock, price rose +$23.54 |
| **Brier score** | 0.032 🟢 |

<details><summary>📰 Oil market context (cutoff: 2026-03-01)</summary>

### **Oil Market Intelligence Briefing: March 1, 2026**

**Immediate Catalyst: The "Black Saturday" Strikes and Geopolitical Escalation**
The primary driver for the coming week is the massive geopolitical shock following the February 28, 2026, coordinated air strikes by U.S. and Israeli forces against Iranian military and nuclear infrastructure. As of today, Sunday, March 1, global markets are bracing for a violent "gap up" at the Monday open. WTI, which closed Friday at approximately $66–$67/bbl, is expected to surge toward the $80–$85 range immediately. The reported assassination of Iran’s Supreme Leader during the strikes has moved the conflict from a regional proxy war to an existential state-level confrontation, placing a massive "war premium" back into crude prices that had previously been trading on bearish oversupply fundamentals.

**Geopolitical Flashpoint: Strait of Hormuz and Shipping Blockades**
The most critical risk to supply is the effective closure of the Strait of Hormuz. Following the strikes, the Iranian Revolutionary Guard Corps (IRGC) has issued warnings forbidding passage through the waterway, which handles roughly 20% of global seaborne oil (approx. 20 mb/d). Major shipping conglomerates, including Maersk and MSC, have already announced the suspension of all transits as of this morning. Reports of sea mines and the boarding of merchant vessels in the Gulf of Oman suggest a total maritime blockade is imminent. Unlike the Red Sea disruptions of 2024, there is no viable alternative for the volume of crude and LNG exiting the Persian Gulf; even a partial 10-day closure could remove 100 million barrels from the global market, potentially pushing WTI toward triple digits within the next 5–10 days.

**OPEC+ Policy: The March 1 Virtual Meeting**
In a move to signal market stability, OPEC+ members (led by Saudi Arabia and Russia) met virtually today, March 1. The group decided to proceed with a scheduled production adjustment of 206,000 barrels per day for April 2026, continuing the gradual unwinding of voluntary cuts. However, the group reaffirmed "full flexibility" to pause or reverse these hikes if market conditions deteriorate. While the production increase is technically bearish, it is being viewed by analysts as a "placeholder" decision; the real focus is on whether Saudi Arabia will activate its East-West Pipeline to bypass the Strait of Hormuz, though its 7 mb/d capacity cannot fully offset a total Gulf blockade.

**US SPR and Energy Policy Signals**
The Trump administration enters this crisis with the Strategic Petroleum Reserve (SPR) at approximately 415.4 million barrels (roughly 58% capacity), following a year of steady refilling efforts. While the administration has signaled a commitment to "energy dominance," the market is hyper-focused on whether an emergency SPR release will be coordinated with the IEA to blunt the expected price spike. Analysts at Goldman Sachs and Morgan Stanley have already revised their March targets, with some projecting Brent could exceed $110/bbl if the Hormuz closure persists. Any signal from the White House regarding a massive "emergency exchange" (loan) of crude could be the only factor capable of preventing WTI from testing its 2022 highs of $120+ in the coming week.

**5–10 Day Risk Assessment: Sudden Move Factors**
*   **Upside (Bullish):** Iranian retaliatory strikes on Saudi Aramco’s "Abqaiq" facility or UAE processing plants; confirmed mining of the Strait of Hormuz; or a formal declaration of *force majeure* by Kuwaiti or Iraqi exporters.
*   **Downside (Bearish):** An immediate U.S.-led "Freedom of Navigation" operation successfully escorting tankers through the Strait; or a surprise announcement of a 100M+ barrel global SPR release.
*   **Trend:** The technical "descending channel" that dominated 2025 has been shattered. Expect extreme volatility with a heavy bullish bias as the market prices in the largest potential supply disruption in history.

</details>



---
### Mar 9, 2026 — WTI $94.77 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **68%**  `███████░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The closure of the Strait of Hormuz has removed 14 mb/d of supply, creating a massive structural deficit that outweighs the minor OPEC+ production increases. WTI has demonstrated extreme momentum, rising over $23 in the last five trading days, making a further $5 move highly probable under current volatility. While a rumored 400-million-barrel IEA release poses a significant downside risk, the immediate 'war premium' and physical shortages are likely to sustain the upward trajectory in the short term. |
| **Key signals** | Strait of Hormuz operational status · IEA emergency release announcement · Integrity of Saudi East-West Pipeline |
| **Actual outcome** | 🟢 **No shock** — price moved **-1.27 /bbl** |
| **Verdict** | ❌ **False alarm** — predicted shock, price moved -1.27 |
| **Brier score** | 0.462 🔴 |

<details><summary>📰 Oil market context (cutoff: 2026-03-08)</summary>

### **Oil Market Intelligence Briefing: March 8, 2026**

**Price Action and Market Sentiment**
As of March 8, 2026, WTI crude is experiencing unprecedented vertical volatility, trading in a wide range between **$125 and $145 per barrel**. This follows a catastrophic price spike initiated in February 2026 after the escalation of the "Iran War" involving U.S. and Israeli forces. The market has transitioned from a period of relative stability in early January (where WTI averaged $65) to a state of extreme "war premium" pricing. Sentiment is dominated by fear of a prolonged global energy deficit, as the effective closure of the Strait of Hormuz has removed nearly **14 million barrels per day (mb/d)** of Middle Eastern supply from the global balance.

**OPEC+ Policy and Supply Outlook**
On March 1, 2026, the eight key OPEC+ members (led by Saudi Arabia and Russia) announced they would increase production by **206,000 bpd starting in April 2026**. While this move was intended to signal a commitment to market stability, traders have largely dismissed it as a "drop in the bucket" given the scale of the current disruption. The group is attempting to resume the unwinding of voluntary cuts first announced in 2023, but the physical inability to export oil from the Persian Gulf has rendered these quotas largely theoretical for several members. Non-OPEC+ production from the Atlantic Basin (U.S., Brazil, and Guyana) is pushing toward record levels, but infrastructure bottlenecks at Gulf Coast terminals are limiting the speed at which this "new" oil can reach hard-hit Asian markets.

**Geopolitical Risks and Shipping Disruptions**
The primary driver of market panic is the **near-total shutdown of the Strait of Hormuz**. Naval engagements and missile strikes have made the waterway impassable for commercial tankers, effectively trapping the output of Saudi Arabia, Kuwait, Iraq, and the UAE. While some oil is being rerouted through Saudi Arabia’s East-West Pipeline to the Red Sea, these alternative routes are operating at maximum capacity and are themselves under threat of drone activity. The U.S. naval blockade of Iranian exports has further tightened the market, though reports suggest that some "dark fleet" volumes continue to move toward China under extreme risk.

**US SPR and Emergency Policy Signals**
The U.S. Strategic Petroleum Reserve (SPR) stands at approximately **412 million barrels**, having been slowly replenished to roughly 58% capacity under the current administration's "Energy Dominance" policy. However, the focus has shifted abruptly from refilling to emergency drawdown. There are intense rumors in Washington and Paris that the **IEA is preparing to announce a coordinated release of up to 400 million barrels**—the largest in history—to prevent a global economic collapse. As of today, March 8, the market is pricing in the high probability of this announcement, which would include a U.S. contribution of roughly 172 million barrels via an emergency exchange program.

**Factors for a Sudden Large Move (Next 5–10 Days)**
*   **IEA Announcement:** A formal confirmation of the 400mb coordinated release could cause a sudden **$15–$20 "relief drop"** in WTI as liquidity returns to the market.
*   **Infrastructure Sabotage:** Any confirmed damage to the East-West Pipeline or major Saudi processing facilities (e.g., Abqaiq) would likely push Brent/WTI toward **$180+** instantly.
*   **Ceasefire Rumors:** Any diplomatic signal regarding a "maritime corridor" or a pause in hostilities between the U.S./Israel and Iran would trigger a massive liquidation of long positions.
*   **Refinery Constraints:** With Middle Eastern crude cut off, complex refineries in Asia may begin "economic shutdowns" due to lack of feedstock, potentially causing a collapse in crude demand that paradoxically sends refined product prices (diesel/jet fuel) even higher while crude softens.

</details>



---
### Mar 16, 2026 — WTI $93.50 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **42%**  `████░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently in a volatile tug-of-war between a historic 10-15 mb/d supply disruption and a massive 400-million-barrel global SPR release. While the SPR intervention has successfully triggered a $20 retracement from recent highs, the risk of military escalation or failed tanker escorts in the Strait of Hormuz remains a high-probability upside catalyst. Given the extreme daily volatility, a $5 move represents a return to the mean of the current conflict-driven price regime. |
| **Key signals** | U.S. Navy tanker escort outcomes · Iranian retaliation against regional infrastructure · SPR liquidity absorption rates |
| **Actual outcome** | 🟢 **No shock** — price moved **-5.37 /bbl** |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved -5.37 |
| **Brier score** | 0.176 🟡 |

<details><summary>📰 Oil market context (cutoff: 2026-03-15)</summary>

### **Oil Market Intelligence Briefing: March 15, 2026**

**Price Action & Market Sentiment**
WTI crude is currently trading near **$96.00/bbl**, while Brent remains elevated above **$103.00/bbl**. The market is characterized by extreme volatility following the launch of a major U.S.-Israeli military operation against Iran on February 28, 2026. Prices initially spiked to a peak of **$113.13** in early March but have recently retraced following a massive emergency intervention by the International Energy Agency (IEA). Despite this slight cooling, the Brent-WTI spread has widened to roughly **$12/bbl**, reflecting the disproportionate impact of Middle Eastern supply disruptions on international benchmarks and surging shipping insurance costs.

**OPEC+ Policy & Supply Outlook**
OPEC+ has maintained its decision to **pause production increases** through the end of Q1 2026, citing seasonal demand weakness and the extreme uncertainty of the current conflict. However, the alliance’s internal cohesion is under severe strain; the **UAE has announced its intention to exit OPEC** effective May 1, 2026, a move that threatens to dismantle the group’s long-term spare capacity management. Currently, an estimated **10–15 million barrels per day (mb/d)** of production from the Persian Gulf is at risk or shut-in due to the effective closure of the Strait of Hormuz, leading to the largest supply shock in modern history.

**Geopolitical Flashpoints: The Persian Gulf War**
The **Strait of Hormuz remains a de facto no-go zone** for commercial tankers following the escalation of hostilities between the U.S.-led coalition and Iran. Reports of underwater mines and drone attacks on shipping have halted approximately 20% of global oil flows. While the U.S. Navy has begun operations to escort commercial vessels, the "tanker war" dynamics have effectively neutralized the export capacity of Iraq, Kuwait, and the UAE. Any sign of Iranian retaliation against Saudi energy infrastructure or a failed escort mission could send prices toward the **$140–$150** range within days.

**US Policy & SPR Intervention**
In a historic move on March 11, 2026, President Trump authorized the release of **172 million barrels** from the Strategic Petroleum Reserve (SPR) as part of a coordinated 400-million-barrel IEA action. To manage the fiscal impact, the Department of Energy is utilizing a "swap" strategy—selling immediate-delivery barrels at current highs ($96+) while simultaneously contracting to repurchase them for 2027 delivery at significantly lower futures prices (~$71). This temporal arbitrage is intended to flood the market with liquidity in the next 120 days, though critics warn that the SPR is now at its lowest level since its inception, leaving the U.S. with a dangerously thin margin for further shocks.

**Short-Term Outlook (5–10 Days)**
The next 5–10 days are critical for WTI. A **sudden large move** is likely depending on two factors:
1. **Escalation/Retaliation:** If Iran successfully strikes Atlantic Basin production or key transit hubs outside the Gulf, WTI will likely break past **$120/bbl**.
2. **Escort Success:** If the first wave of U.S.-escorted tankers successfully transits the Strait of Hormuz without incident, a "relief sell-off" could see WTI drop toward **$85/bbl** as the immediate scarcity premium evaporates. 
Traders should also watch for an emergency OPEC+ meeting, as the group faces immense pressure to fill the void left by the Persian Gulf blockade.

</details>



---
### Mar 23, 2026 — WTI $88.13 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **40%**  `████░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently undergoing a sharp correction as it absorbs the impact of a historic 400-million-barrel coordinated SPR release, which has temporarily offset the panic from the Hormuz blockade. However, the underlying deficit of 20 million b/d remains fundamentally unresolved, and the recent $10 drop to $88.13 may be viewed as an overcorrection by some participants. A $5 bounce is highly plausible if the physical delivery of SPR barrels faces logistical hurdles or if military actions expand to inland infrastructure. |
| **Key signals** | SPR discharge and delivery logistics · Potential military strikes on Saudi Ghawar or Abqaiq infrastructure · Emerging demand destruction data from major Asian importers |
| **Actual outcome** | 🔴 **SHOCK** — price moved **+14.75 /bbl** |
| **Verdict** | ❌ **Missed** — predicted no shock, price rose +$14.75 |
| **Brier score** | 0.360 🔴 |

<details><summary>📰 Oil market context (cutoff: 2026-03-22)</summary>

### **Oil Market Intelligence Briefing**
**Date:** March 22, 2026
**Subject:** Global Supply Crisis and Strategic Intervention

#### **Price Action and Immediate Context**
WTI and Brent crude prices are currently trading at extreme premiums following the most severe supply shock in modern history. Since the initiation of U.S.-Israeli military action against Iran on February 28 and the subsequent retaliatory closure of the Strait of Hormuz on March 4, Brent has surged past the **$120/bbl** mark, with WTI trailing closely in the **$110–$115/bbl** range. Market volatility is at decadal highs as traders price in the total loss of approximately **20 million barrels per day (b/d)** of crude and refined products that typically transit the Strait. While prices have stabilized slightly from their initial peak due to intervention news, the "war-risk premium" remains the dominant driver of the current curve.

#### **Supply Disruptions and OPEC+ Status**
The physical market is in a state of emergency. Gulf producers, including Saudi Arabia, Iraq, Kuwait, and the UAE, have been forced to shut in an estimated **10 million b/d** of production as storage capacity in the region reaches its limits with no viable export outlets. QatarEnergy has declared *force majeure* on all LNG and condensate exports. OPEC+ as a cohesive entity is effectively paralyzed; while the group had previously agreed to extend production cuts through 2026, those quotas are now irrelevant compared to the massive involuntary shut-ins. Furthermore, the **UAE’s recent announcement to exit OPEC** (effective May 1) has introduced a secondary layer of long-term structural uncertainty regarding future spare capacity and cartel unity.

#### **Geopolitical Risks and Shipping**
The Strait of Hormuz remains functionally closed to commercial traffic. War-risk insurance premiums for the Persian Gulf have become prohibitive, and several major tanker fleets have suspended all regional operations. Beyond the blockade, there are credible reports of damage to regional refining infrastructure—totaling roughly **3 million b/d** of capacity—due to asymmetric attacks. Shipping lanes in the Red Sea are also seeing increased congestion as Saudi Arabia attempts to divert limited volumes through the East-West Pipeline to Yanbu, though this route can only mitigate a fraction of the lost Hormuz volumes.

#### **Strategic Reserve Intervention**
The primary bearish counterweight is the massive, coordinated emergency release announced by the IEA on March 11. Member nations have committed to releasing **400 million barrels** from strategic stocks, the largest such action in history. The U.S. portion of this release, totaling **172 million barrels**, is scheduled to begin hitting the market this week. While this provides a temporary physical buffer, analysts warn that the U.S. Strategic Petroleum Reserve (SPR) is being drawn down from an already depleted level of approximately **415 million barrels**, potentially leaving the global market vulnerable if the conflict extends beyond a 90-day window.

#### **Short-Term Volatility Catalysts (5–10 Day Outlook)**
WTI is susceptible to a **$15–$20 move** in either direction over the next 10 days based on the following triggers:
*   **Escalation/De-escalation Signals:** Any indication of a ceasefire or, conversely, a widening of the conflict to include strikes on Saudi "Ghawar" or "Abqaiq" infrastructure would cause immediate price gaps.
*   **SPR Delivery Logistics:** The market is hyper-focused on the actual "discharge rate" of the 172M barrel U.S. release. Any logistical delays in moving these barrels to Gulf Coast refiners will cause a sharp upward correction in WTI.
*   **Demand Destruction Data:** Early indicators of "panic-driven" demand destruction in Asia and Europe are emerging. If March 2026 consumption data shows a contraction greater than the current 1M b/d estimate, prices may see a sharp "relief" sell-off.

</details>


In [12]:
# ── Binary comparison chart ──────────────────────────────────────────────
METHOD_COLORS = {"Prophet": CLR_PROPHET, "Analyst Agent": CLR_AGENT}
METHOD_SYMBOL = {"Prophet": "square", "Analyst Agent": "circle"}
METHOD_DASH = {"Prophet": "dot", "Analyst Agent": "solid"}

origins_ordered = [o.strftime("%Y-%m-%d") for o in SHOCK_ORIGINS]
outcome_by_key = {
    key: int(binary_df[(binary_df["origin"] == key) & (binary_df["method"] == "Prophet")].iloc[0]["outcome"])
    for key in origins_ordered
}
shock_indices = [i for i, k in enumerate(origins_ordered) if outcome_by_key[k] == 1]

fig = psp.make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.58, 0.42],
    vertical_spacing=0.20,
    subplot_titles=[
        "P(WTI up > +$5 /bbl in 5 trading days)",
        "Cumulative mean Brier score (lower = better)",
    ],
)

for i in shock_indices:
    for row_n, (y0, y1) in [(1, (-0.12, 1.08)), (2, (0.0, 0.30))]:
        fig.add_shape(
            type="rect",
            layer="below",
            xref=f"x{'' if row_n == 1 else row_n}",
            yref=f"y{'' if row_n == 1 else row_n}",
            x0=i - 0.48,
            x1=i + 0.48,
            y0=y0,
            y1=y1,
            fillcolor="rgba(214,39,40,0.12)",
            line_width=0,
        )
    fig.add_annotation(
        x=origins_ordered[i],
        y=1.06,
        text="<b>SHOCK</b>",
        showarrow=False,
        font={"size": 9, "color": CLR_CONFLICT},
        xref="x",
        yref="y",
    )

for method in ["Analyst Agent", "Prophet"]:
    sub = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    fig.add_trace(
        go.Scatter(
            x=sub["origin"],
            y=sub["prob"],
            name=method,
            mode="lines+markers",
            line={"color": METHOD_COLORS[method], "width": 2.5, "dash": METHOD_DASH[method]},
            marker={"size": 10, "symbol": METHOD_SYMBOL[method]},
            legendgroup=method,
            showlegend=True,
            hovertemplate="%{x}<br>P(up)=%{y:.0%}<extra>" + method + "</extra>",
        ),
        row=1,
        col=1,
    )
    yshift = 12 if method == "Analyst Agent" else -14
    for _, r in sub.iterrows():
        fig.add_annotation(
            x=r["origin"],
            y=r["prob"],
            text=f"{r['prob']:.0%}",
            showarrow=False,
            font={"size": 8, "color": METHOD_COLORS[method]},
            yshift=yshift,
            row=1,
            col=1,
        )

fig.add_hline(y=0.5, line={"color": "#d0d0d0", "dash": "dot", "width": 1.2}, row=1, col=1)

for method in ["Analyst Agent", "Prophet"]:
    sub = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    cum_brier = sub["brier"].expanding().mean().values
    fig.add_trace(
        go.Scatter(
            x=sub["origin"].values,
            y=cum_brier,
            name=method,
            mode="lines+markers",
            line={"color": METHOD_COLORS[method], "width": 2.5, "dash": METHOD_DASH[method]},
            marker={"size": 8, "symbol": METHOD_SYMBOL[method]},
            legendgroup=method,
            showlegend=False,
            hovertemplate="%{x}<br>Cumul. Brier: %{y:.3f}<extra>" + method + "</extra>",
        ),
        row=2,
        col=1,
    )

fig.add_hline(
    y=0.25,
    line={"color": "#aaa", "dash": "dot", "width": 1.5},
    annotation_text="0.25 random ceiling",
    annotation_position="top right",
    annotation_font={"size": 9, "color": "#888"},
    row=2,
    col=1,
)
fig.update_layout(
    title={"text": "Analyst Agent vs. Prophet — Upward Shock (Feb–Mar 2026)", "x": 0.5, "font": {"size": 13}},
    height=520,
    width=640,
    template="plotly_white",
    xaxis={"type": "category", "tickangle": -35, "showgrid": False},
    xaxis2={"type": "category", "tickangle": -35, "showgrid": False},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.04, "xanchor": "right", "x": 1, "font": {"size": 11}},
    margin={"t": 80, "b": 65, "l": 60, "r": 35},
)
fig.update_yaxes(title_text="P(up > +$5)", range=[-0.08, 1.14], tickformat=".0%", dtick=0.25, row=1, col=1)
fig.update_yaxes(title_text="Brier score", range=[0, 0.30], row=2, col=1)
fig.update_yaxes(showgrid=True, gridcolor="#ececec", gridwidth=0.7)
fig.show()

In [13]:
# ── Brier score summary table ────────────────────────────────────────────
pivot = binary_df.assign(
    prob_str=lambda d: d["prob"].map(lambda p: f"{p:.0%}"),
    brier_str=lambda d: d["brier"].map(lambda b: f"{b:.3f}"),
    summary=lambda d: d["prob_str"] + "  (BS " + d["brier_str"] + ")",
).pivot_table(index="origin", columns="method", values="summary", aggfunc="first")
pivot.index.name = "Origin"
pivot.columns.name = None

meta = binary_df.groupby("origin").first()[["outcome", "delta"]].reindex(pivot.index)
meta["delta"] = meta["delta"].map("{:+.2f}".format)
meta.columns = ["Shock? (1=yes)", "5-day Δ ($)"]

col_order = ["Shock? (1=yes)", "5-day Δ ($)"] + [c for c in ["Prophet", "Analyst Agent"] if c in pivot.columns]
display(pd.concat([meta, pivot], axis=1)[col_order])

mean_brier = (
    binary_df.groupby("method")["brier"]
    .mean()
    .reindex(["Prophet", "Analyst Agent"])
    .rename("Mean Brier score")
    .map("{:.4f}".format)
    .to_frame()
)
mean_brier.index.name = "Method"
print("\nMean Brier score (lower = better, 0.25 = random ceiling):")
display(mean_brier)

,Shock? (1=yes),5-day Δ ($),Prophet,Analyst Agent
Origin,,,,
2026-02-02,0,+2.22,3% (BS 0.001),12% (BS 0.014)
2026-02-09,0,-2.03,2% (BS 0.000),22% (BS 0.048)
2026-02-17,0,+3.30,9% (BS 0.008),18% (BS 0.032)
2026-02-23,0,+4.92,3% (BS 0.001),38% (BS 0.144)
2026-03-02,1,+23.54,0% (BS 0.996),82% (BS 0.032)
2026-03-09,0,-1.27,0% (BS 0.000),68% (BS 0.462)
2026-03-16,0,-5.37,0% (BS 0.000),42% (BS 0.176)
2026-03-23,1,+14.75,0% (BS 1.000),40% (BS 0.360)



Mean Brier score (lower = better, 0.25 = random ceiling):


,Mean Brier score
Method,
Prophet,0.2509
Analyst Agent,0.1588


---
## Task C — Scenario Analysis

The same agent. The same system prompt. A different task specification.

Instead of a numeric prediction or a probability, we ask: *“What are the three scenarios that energy analysts are most actively debating for WTI crude over the next 60 days?”* For each scenario the agent names it, describes it, assigns a probability, gives a conditional price estimate at 60 days, and identifies the key drivers.

This is closer to the work of a real analyst than a point forecast — it surfaces the distribution of expert opinion rather than collapsing it to a single number. There is no ground truth to score against: the value is in the structured reasoning.

> **Origin:** Mar 2, 2026 — conflict onset in the Strait of Hormuz. Market context reused from Task A.

In [14]:
# ── Run or load Task C — Scenario Analysis ───────────────────────────────────
# Context for Mar-2 was already retrieved for Task A; we reuse it here.

SCENARIO_ORIGIN = "2026-03-02"
origin_price_c = float(price_df[price_df.index >= pd.Timestamp(SCENARIO_ORIGIN)].iloc[0]["price"])

if SCENARIO_CACHE.exists():
    with open(SCENARIO_CACHE) as f:
        scenario_result: dict = json.load(f)
    print(f"Loaded cached scenario forecast for {SCENARIO_ORIGIN}.")
else:
    print(f"Running Task C — Scenario Analysis  (origin: {SCENARIO_ORIGIN}, WTI ${origin_price_c:.2f})")
    ctx_c = traj_contexts[SCENARIO_ORIGIN]  # reuse context retrieved for Task A
    hist_c = compress_history(price_df, pd.Timestamp(SCENARIO_ORIGIN))
    hist_str = serialize_history(hist_c, precision=2)
    scenario_result = run_analyst(TASK_SCENARIOS, hist_str, ctx_c, SCENARIO_ORIGIN, origin_price_c)
    with open(SCENARIO_CACHE, "w") as f:
        json.dump(scenario_result, f, indent=2)
    print("Saved to cache.")

# ── I/O inspection ────────────────────────────────────────────────────────────
display(
    MD(
        "### Task C — I/O Inspection: " + SCENARIO_ORIGIN + "\n\n"
        "Same agent, same system prompt — a third task specification in the user message."
    )
)

ctx_c_preview = traj_contexts[SCENARIO_ORIGIN][:600].strip() + " ..."
hist_c2 = compress_history(price_df, pd.Timestamp(SCENARIO_ORIGIN))
hist_lines_c = serialize_history(hist_c2, precision=2).split("\n")
task_preview_c = TASK_SCENARIOS.replace("\n", "\n> ")

sys_preview = _ANALYST_SYSTEM.replace("\n", "\n> ")
display(
    MD(
        "<details><summary>📋 <strong>System prompt</strong> — same as Tasks A and B</summary>\n\n"
        "> " + sys_preview + "\n\n</details>"
    )
)

user_display_c = (
    f"**Price history** — last 10 of {len(hist_lines_c)} rows:\n\n"
    "```\n" + "\n".join(hist_lines_c[-10:]) + "\n```\n\n"
    f"**Market briefing** (first 600 chars, cutoff: {(pd.Timestamp(SCENARIO_ORIGIN) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}):\n\n"
    + ctx_c_preview
    + "\n\n"
    "**TASK SPECIFICATION** (Task C — Scenario Analysis):\n\n"
    "> " + task_preview_c
)
display(
    MD(
        "<details><summary>📤 <strong>User message</strong> (abbreviated)</summary>\n\n"
        + user_display_c
        + "\n\n</details>"
    )
)

Running Task C — Scenario Analysis  (origin: 2026-03-02, WTI $71.23)


Saved to cache.


### Task C — I/O Inspection: 2026-03-02

Same agent, same system prompt — a third task specification in the user message.

<details><summary>📋 <strong>System prompt</strong> — same as Tasks A and B</summary>

> You are an expert oil market analyst.
> 
> You will receive:
>   1. WTI crude oil price history (daily, most recent last)
>   2. An oil market intelligence briefing with a strict temporal cutoff
>   3. A TASK SPECIFICATION that defines the exact question and the required
>      JSON output schema
> 
> Read the data and briefing carefully, then execute the task precisely.
> Return ONLY valid JSON that matches the schema described in the task specification.
> Output ONLY the JSON object — no preamble, no explanation outside the JSON.

</details>

<details><summary>📤 <strong>User message</strong> (abbreviated)</summary>

**Price history** — last 10 of 368 rows:

```
2026-02-17 $62.33
2026-02-18 $65.19
2026-02-19 $66.43
2026-02-20 $66.39
2026-02-23 $66.31
2026-02-24 $65.63
2026-02-25 $65.42
2026-02-26 $65.21
2026-02-27 $67.02
2026-03-02 $71.23
```

**Market briefing** (first 600 chars, cutoff: 2026-03-01):

### **Oil Market Intelligence Briefing: March 1, 2026**

**Price Action and Immediate Trend**
As of March 1, 2026, the global oil market is entering a state of extreme volatility following the dramatic escalation of hostilities in the Persian Gulf over the last 24 hours. Prior to the weekend, WTI was trending modestly upward, closing on February 27 at approximately **$65.99**, with Brent near **$71.58**. However, the launch of "Operation Epic Fury" by U.S. and Israeli forces on February 28—which reportedly targeted Iranian military and nuclear infrastructure—has triggered a massive "black swan ...

**TASK SPECIFICATION** (Task C — Scenario Analysis):

> Identify the three scenarios that oil market analysts and experts are most
> actively debating for WTI crude over the next 60 days, given the current
> market context and price history.
> 
> For each scenario:
>   - Give it a concise name (3-6 words)
>   - Describe it in 1-2 sentences
>   - Assign a probability (all three must sum to <= 1.0)
>   - Provide an expected WTI price range at the 60-day horizon as [low, high]
>   - Give your point estimate for WTI at 60 days under this scenario
>   - List 1-2 key drivers that would cause this scenario to materialise
> 
> Also identify which scenario is the base case and provide an overall
> one-paragraph reasoning summary.
> 
> Return JSON with exactly these fields:
> {
>   "scenarios": [
>     {
>       "name":               "<string>",
>       "description":        "<string>",
>       "probability":        <float>,
>       "wti_range_60d":      [<float_low>, <float_high>],
>       "point_estimate_60d": <float>,
>       "key_drivers":        ["<driver 1>", "<driver 2>"]
>     }
>   ],
>   "base_case":  "<scenario name>",
>   "reasoning":  "<paragraph>"
> }

</details>

In [15]:
# ── Display Task C results — scenario cards ──────────────────────────────────
def _prob_bar(p: float, width: int = 10) -> str:
    n = max(0, min(width, int(round(p * width))))
    return "\u2588" * n + "\u2591" * (width - n)


scenarios = scenario_result.get("scenarios", [])
base_case = scenario_result.get("base_case", "")
reasoning = scenario_result.get("reasoning", "")

display(MD(f"#### 📥 Agent response — Task C  *(as of {SCENARIO_ORIGIN}, WTI ${origin_price_c:.2f})*"))

for s in scenarios:
    name = s.get("name", "?")
    desc = s.get("description", "")
    prob = float(s.get("probability", 0))
    rng = s.get("wti_range_60d", [float("nan"), float("nan")])
    lo, hi = float(rng[0]), float(rng[1])
    pe = float(s.get("point_estimate_60d", float("nan")))
    drivers = s.get("key_drivers", [])
    is_base = "  \u2b50 **base case**" if name == base_case else ""

    display(
        MD(
            f"---\n"
            f"**{name}**{is_base}\n\n"
            f"{desc}\n\n"
            f"| | |\n|---|---|\n"
            f"| Probability | **{prob:.0%}**  `{_prob_bar(prob)}` |\n"
            f"| WTI range at 60 days | ${lo:.0f} \u2013 ${hi:.0f} /bbl |\n"
            f"| Point estimate | **${pe:.0f} /bbl** |\n"
            f"| Key drivers | {' \u00b7 '.join(drivers) if drivers else '\u2014'} |\n"
        )
    )

display(MD(f"---\n\n> **Overall reasoning:** {reasoning}"))

#### 📥 Agent response — Task C  *(as of 2026-03-02, WTI $71.23)*

---
**Naval Intervention and SPR Release**  ⭐ **base case**

A US-led naval coalition successfully reopens the Strait of Hormuz within two weeks, while a massive coordinated SPR release offsets the temporary supply gap.

| | |
|---|---|
| Probability | **45%**  `████░░░░░░` |
| WTI range at 60 days | $68 – $82 /bbl |
| Point estimate | **$74 /bbl** |
| Key drivers | Successful US naval escort operations in the Persian Gulf · Emergency SPR release of 2 million barrels per day |


---
**Prolonged Blockade and Regional Escalation**

The Strait remains closed for over 14 days and Iranian retaliatory strikes cause physical damage to Saudi or Emirati export infrastructure.

| | |
|---|---|
| Probability | **30%**  `███░░░░░░░` |
| WTI range at 60 days | $105 – $135 /bbl |
| Point estimate | **$118 /bbl** |
| Key drivers | Sustained closure of the Strait of Hormuz · Direct missile or drone hits on major Gulf oil terminals |


---
**Partial Blockade and High War Premium**

Shipping resumes under extreme risk with high insurance costs, while ongoing regional tensions prevent a return to pre-conflict price levels.

| | |
|---|---|
| Probability | **25%**  `██░░░░░░░░` |
| WTI range at 60 days | $85 – $102 /bbl |
| Point estimate | **$91 /bbl** |
| Key drivers | Persistent 'war premium' in maritime insurance markets · Limited bypass capacity via the East-West Pipeline to Yanbu |


---

> **Overall reasoning:** The market is currently in a state of 'black swan' repricing following the escalation of Operation Epic Fury. While the immediate gap-up to $71.23 reflects the shock of the Hormuz blockade, the base case assumes that the strategic importance of the Strait will compel a rapid and decisive military response from the U.S. and its allies to restore transit. Furthermore, the 415 million barrels in the U.S. SPR provide a significant buffer that, when deployed, historically dampens localized WTI volatility. Unless physical infrastructure in Saudi Arabia is permanently disabled, the market is likely to see a 'spike and fade' pattern where the initial panic is mitigated by emergency supply measures and naval intervention, though a higher floor will be established compared to the pre-conflict $60/bbl forecasts.

---
## Summary

All three tasks run through the same two-stage pipeline and the same Analyst Agent configuration. Only the text in the user message changes.

**Task A — Trajectory:** Prophet predicts mean reversion from every origin. The Agent, reading conflict news, revises sharply upward from the Mar-2 origin ($71 → day-21 estimate $96). Mean MAE: Prophet $15.25, Agent $4.27.

**Task B — Binary shock:** Prophet assigns near-zero probability to further upside from elevated prices. The Agent correctly elevated P(up) to ~82% on Mar-2 — the week prices surged $23. Cumulative Brier: Prophet 0.2509 (≈ random), Agent 0.1588.

**Task C — Scenario analysis:** Rather than a single number, the Agent articulates the distribution of expert opinion: three named scenarios with probabilities, conditional price ranges, and key drivers. No ground truth to score against — but this is exactly the kind of structured qualitative output that supports real-world decision-making.

**The architectural insight:** a generic system prompt + task-specific user messages keeps the agent configuration stable while the task surface is unlimited. This is the design principle the bootcamp is built on: a reusable agentic forecasting platform that participants can extend to any domain, question type, or data source.